In [ ]:
import os
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version used by PyTorch:", torch.version.cuda)

dataset_path = "/kaggle/input"

print(f"\nTop-level contents of: {dataset_path}\n")
for name in os.listdir(dataset_path):
    full = os.path.join(dataset_path, name)
    if os.path.isdir(full):
        try:
            files = os.listdir(full)
            print(f"{name}: {len(files)} items")
        except Exception as e:
            print(f"{name}: error reading ({e})")
    else:
        print(name)

In [ ]:
import os
from collections import defaultdict

input_dir = "/kaggle/input"
samples = defaultdict(list)
limits = {"Mendeley": 30, "BanglaFake": 30, "Other": 30}

def done(samples, limits):
    return all(len(samples[k]) >= limits[k] for k in limits)

for root, dirs, files in os.walk(input_dir):
    if done(samples, limits):
        break

    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        lower = file_path.lower()

        if "bangla audio dataset" in lower or "mendeley" in lower:
            key = "Mendeley"
        elif "banglafake" in lower or "bengali-deepfake-forensics-dataset" in lower:
            key = "BanglaFake"
        else:
            key = "Other"

        if len(samples[key]) < limits[key]:
            samples[key].append(file_path)

print("=== SAMPLE FILE PATHS BY DATASET ===\n")
for key in ["Mendeley", "BanglaFake", "Other"]:
    paths = samples[key]
    print(f"\n--- {key} ({len(paths)} samples shown) ---")
    for p in paths:
        print(p)

In [ ]:
from collections import Counter

input_dir = "/kaggle/input"
folder_counter = Counter()

max_dirs = 2000  # safety limit
dir_count = 0

for root, dirs, files in os.walk(input_dir):
    dir_count += 1
    if dir_count > max_dirs:
        break

    parts = root.split(os.sep)
    for part in parts:
        folder_counter[part] += 1

print("=== MOST COMMON FOLDER NAMES ===")
for name, count in folder_counter.most_common(50):  # reduce noise
    print(f"{name}: {count}")

In [ ]:
import os

print("=== BANGLAFAKE ROOTS WITH WAV FILES ===\n")

seen = set()
max_hits = 50  # prevent over-scanning

for root, dirs, files in os.walk("/kaggle/input"):
    lower = root.lower()

    # Filter FIRST (important)
    if "banglafake" not in lower and "bengali-deepfake-forensics-dataset" not in lower:
        continue

    # Then check for wav
    for f in files:
        if f.lower().endswith(".wav"):
            if root not in seen:
                print(root)
                seen.add(root)
                break  # stop checking more files in this folder

    if len(seen) >= max_hits:
        break

Create the Master Metadata Table

In [ ]:
import os
import pandas as pd

rows = []
input_dir = "/kaggle/input"

for root, dirs, files in os.walk(input_dir):
    for file in files:
        if not file.lower().endswith(".wav"):
            continue

        file_path = os.path.join(root, file)
        lower_path = file_path.lower()
        path_parts = lower_path.split(os.sep)

        source = None
        sub_source = "Unknown"
        label = -1

        # =========================================================
        # 1. MENDELEY DATASET
        # =========================================================
        if "bangla audio dataset" in lower_path:
            source = "Mendeley"
            sub_source = "Mendeley"

            if "real" in path_parts:
                label = 0
            elif "fake" in path_parts:
                label = 1

        # =========================================================
        # 2. BANGLAFAKE DATASET
        # =========================================================
        elif "final_data" in path_parts:
            source = "BanglaFake"

            if "deepfake_data_sust" in path_parts:
                sub_source = "SUST"
            elif "deepfake_data_mozilla" in path_parts:
                sub_source = "Mozilla"
            elif "deepfake_data_news" in path_parts:
                sub_source = "Unknown"

            if "real_wav" in path_parts:
                label = 0
            elif "deepfake_wav" in path_parts:
                label = 1

        if source is not None:
            rows.append((file_path, source, sub_source, label))

# Create DataFrame
df = pd.DataFrame(rows, columns=["file_path", "source", "sub_source", "label"])

# -----------------------------
# BEFORE FILTERING (DEBUG)
# -----------------------------
print("\n--- BEFORE FILTERING ---")
print(df.groupby(["source", "label"], observed=True).size().unstack(fill_value=0))

print("\n--- BanglaFake Sub-Source Breakdown ---")
print(df.loc[df["source"] == "BanglaFake", "sub_source"].value_counts())

print("\n--- BanglaFake Sub-Source x Label ---")
print(
    df.loc[df["source"] == "BanglaFake"]
      .groupby(["sub_source", "label"], observed=True)
      .size()
      .unstack(fill_value=0)
)

print("\n--- Mendeley Label Breakdown ---")
print(df.loc[df["source"] == "Mendeley", "label"].value_counts())

# -----------------------------
# CLEANING (NO unnecessary copy)
# -----------------------------
mask = (df["label"] != -1) & ~(
    (df["source"] == "BanglaFake") & (df["sub_source"] == "Unknown")
)
df = df.loc[mask]

# -----------------------------
# MEMORY OPTIMIZATION
# -----------------------------
df["source"] = df["source"].astype("category")
df["sub_source"] = df["sub_source"].astype("category")
df["label"] = df["label"].astype("int8")

# -----------------------------
# FINAL SUMMARY
# -----------------------------
print("\n--- FINAL DATA SUMMARY ---")
print(df.groupby(["source", "label"], observed=True).size().unstack(fill_value=0))

print("\n--- FINAL SUB-SOURCE BREAKDOWN ---")
print(df.groupby(["source", "sub_source", "label"], observed=True).size())

# Save
df.to_csv("master_metadata.csv", index=False)

print("\nSuccess: master_metadata.csv created.")

**Map the Files to Your Protocol**
It will automatically detect which files are "Mozilla" (Diverse) and which are "SUST" (Studio) based on the LJ Speech naming format used in BanglaFake.

In [ ]:
import os
import gc
import pandas as pd

BASE_DIR = "/kaggle/input/bengali-deepfake-forensics-dataset"
BANGLA_FAKE_DIR = os.path.join(BASE_DIR, "final_data")
MENDELEY_DIR = os.path.join(
    BASE_DIR,
    "Bangla Audio Dataset Original and DeepFake Voices for AI-Based Voice Analysis and Detection",
    "Bangla Audio Dataset Original and DeepFake Voices for AI-Based Voice Analysis and Detection",
    "Dataset"
)

rows = []

# =========================================================
# 1. Process BanglaFake
#    SUST -> training/validation
#    Mozilla -> independent test
#    News -> excluded
# =========================================================
for root, dirs, files in os.walk(BANGLA_FAKE_DIR):
    root_lower = root.lower()

    if "deepfake_data_news" in root_lower:
        continue

    if "deepfake_data_mozilla" in root_lower:
        usage = "Independent_Test"
        subset = "Mozilla"
    elif "deepfake_data_sust" in root_lower:
        usage = "Train_Validation"
        subset = "SUST"
    else:
        continue

    if "real_wav" in root_lower:
        label = 0
    elif "deepfake_wav" in root_lower:
        label = 1
    else:
        continue

    for file in files:
        if file.lower().endswith(".wav"):
            full_path = os.path.join(root, file)
            rows.append((full_path, label, "BanglaFake", subset, usage))

# =========================================================
# 2. Process Mendeley
#    Cross-dataset test
# =========================================================
for root, dirs, files in os.walk(MENDELEY_DIR):
    root_lower = root.lower()
    root_parts = root_lower.split(os.sep)

    if "real" in root_parts:
        label = 0
    elif "fake" in root_parts:
        label = 1
    else:
        continue

    for file in files:
        if file.lower().endswith(".wav"):
            full_path = os.path.join(root, file)
            rows.append((full_path, label, "Mendeley", "Mendeley", "Cross_Dataset_Test"))

# Create DataFrame
df = pd.DataFrame(rows, columns=["path", "label", "source", "subset", "usage"])

# Free Python list memory
del rows
gc.collect()

# Memory optimization
df["label"] = df["label"].astype("int8")
df["source"] = df["source"].astype("category")
df["subset"] = df["subset"].astype("category")
df["usage"] = df["usage"].astype("category")

# =========================================================
# SUMMARY
# =========================================================
print("--- TABLE I: EXPERIMENTAL PROTOCOL BREAKDOWN ---")
summary = df.groupby(["usage", "label"], observed=True).size().unstack(fill_value=0)
summary.columns = ["Real (0)", "Fake (1)"]
summary["Total"] = summary.sum(axis=1)
print(summary)

print("\n--- SUBSET BREAKDOWN ---")
subset_summary = df.groupby(["source", "subset", "label"], observed=True).size().unstack(fill_value=0)
subset_summary.columns = ["Real (0)", "Fake (1)"]
subset_summary["Total"] = subset_summary.sum(axis=1)
print(subset_summary)

# Save
df.to_csv("master_metadata_v2.csv", index=False)

print("\nSuccess: master_metadata_v2.csv created.")

Create Logical Shortcuts

In [ ]:
import shutil

# 1. Define original paths (FIXED: direct correct targets)
original_mendeley = "/kaggle/input/bengali-deepfake-forensics-dataset/Bangla Audio Dataset Original and DeepFake Voices for AI-Based Voice Analysis and Detection/Bangla Audio Dataset Original and DeepFake Voices for AI-Based Voice Analysis and Detection/Dataset"

# IMPORTANT: keep this exactly as final_data (correct)
original_banglafake = "/kaggle/input/bengali-deepfake-forensics-dataset/final_data"

# 2. Define clean names
working_dir = "/kaggle/working"
new_mendeley_path = os.path.join(working_dir, "Mendeley_Data")
new_banglafake_path = os.path.join(working_dir, "BanglaFake_Data")

# 3. Safe symlink creation
for target, link_name in [
    (original_mendeley, new_mendeley_path),
    (original_banglafake, new_banglafake_path),
]:
    try:
        if os.path.lexists(link_name):
            if os.path.islink(link_name) or os.path.isfile(link_name):
                os.unlink(link_name)
            elif os.path.isdir(link_name):
                shutil.rmtree(link_name)

        if os.path.exists(target):
            os.symlink(target, link_name)
            print(f"Success: {os.path.basename(link_name)} linked.")
        else:
            print(f"Error: Target path not found: {target}")

    except Exception as e:
        print(f"Error creating link for {link_name}: {e}")

# 4. Final verification (LIGHTWEIGHT)
print("\n--- CLEAN PATHS FOR FORENSIC PIPELINE ---")
print(f"Mendeley Path:   {new_mendeley_path}")
print(f"BanglaFake Path: {new_banglafake_path}")

# Show small sample instead of full count
if os.path.exists(new_banglafake_path):
    print("Sample items:", os.listdir(new_banglafake_path)[:5])

Metadata Mapping

In [ ]:
import os
import gc
import pandas as pd

# Define paths (FIXED: correct structure)
BANGLA_FAKE_DIR = os.path.join("/kaggle/working/BanglaFake_Data", "final_data")
MENDELEY_DIR = "/kaggle/working/Mendeley_Data"

rows = []

# =========================================================
# 1. BANGLAFAKE (exclude news)
# =========================================================
print("Scanning BanglaFake folders...")

for root, dirs, files in os.walk(BANGLA_FAKE_DIR):
    root_lower = root.lower()

    # Skip unwanted folders early
    if "news" in root_lower:
        continue

    # Identify subset + usage
    if "mozilla" in root_lower:
        source_group = "mozilla"
        usage = "Test_Independent_Mozilla"
    elif "sust" in root_lower:
        source_group = "sust"
        usage = "Train_SUST"
    else:
        continue

    # Label detection
    if "real_wav" in root_lower:
        label = 0
    elif "deepfake_wav" in root_lower:
        label = 1
    else:
        continue

    for file in files:
        if file.endswith(".wav"):
            rows.append((
                os.path.join(root, file),
                label,
                "BanglaFake",
                source_group,
                usage
            ))

# =========================================================
# 2. MENDELEY
# =========================================================
print("Scanning Mendeley folders...")

for root, dirs, files in os.walk(MENDELEY_DIR):
    root_lower = root.lower()

    if "real" in root_lower:
        label = 0
    elif "fake" in root_lower:
        label = 1
    else:
        continue

    for file in files:
        if file.endswith(".wav") and not file.startswith("."):
            rows.append((
                os.path.join(root, file),
                label,
                "Mendeley",
                "Mendeley",
                "Test_Cross_Dataset"
            ))

# Create DataFrame
df = pd.DataFrame(rows, columns=["path", "label", "dataset", "source_group", "usage"])

del rows
gc.collect()

# =========================================================
# MEMORY OPTIMIZATION
# =========================================================
df["label"] = df["label"].astype("int8")
df["dataset"] = df["dataset"].astype("category")
df["source_group"] = df["source_group"].astype("category")
df["usage"] = df["usage"].astype("category")

# =========================================================
# SUMMARY
# =========================================================
print("\n--- TABLE I: FINAL FORENSIC DATASET PROTOCOL ---")

if not df.empty:
    summary = df.groupby(['usage', 'source_group', 'label'], observed=True).size().unstack(fill_value=0)
    summary.columns = ['Real (0)', 'Fake (1)']
    summary['Total'] = summary.sum(axis=1)
    print(summary)

    df.to_csv('/kaggle/working/metadata_final.csv', index=False)

    print(f"\nTotal valid files: {len(df)}")
    print("News folder excluded. metadata_final.csv saved!")

else:
    print("Error: No files found. Check symlinks.")

# audio-level statistics

In [ ]:
import gc
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import pandas as pd

# Load latest metadata
csv_path = Path("/kaggle/working/master_metadata_v2.csv")
df = pd.read_csv(csv_path)

print(f"Total audio files to profile: {len(df)}")

# =========================================================
# Compute duration & sample rate
# =========================================================
durations = []
sample_rates = []

paths = df["path"].tolist()

print("Profiling audio files...")
for idx, audio_path in enumerate(paths):
    try:
        sr = librosa.get_samplerate(audio_path)
        duration = librosa.get_duration(path=audio_path)
    except Exception:
        sr = pd.NA
        duration = pd.NA

    sample_rates.append(sr)
    durations.append(duration)

    if (idx + 1) % 5000 == 0:
        print(f"Processed {idx + 1} files...")

df["duration_sec"] = pd.to_numeric(durations, errors="coerce")
df["sampling_rate"] = pd.to_numeric(sample_rates, errors="coerce")

# Remove unreadable files
df = df.dropna(subset=["duration_sec", "sampling_rate"]).reset_index(drop=True)
df["sampling_rate"] = df["sampling_rate"].astype("int32")
gc.collect()

# =========================================================
# Summary statistics
# =========================================================
summary_table = (
    df.groupby(["usage", "label"], observed=True)
      .agg(
          count=("duration_sec", "count"),
          mean_dur=("duration_sec", "mean"),
          median_dur=("duration_sec", "median"),
          min_dur=("duration_sec", "min"),
          max_dur=("duration_sec", "max"),
          unique_sr=("sampling_rate", "nunique"),
          mode_sr=("sampling_rate", lambda x: x.mode().iat[0] if not x.mode().empty else pd.NA),
      )
      .round(2)
)

print("\n--- TABLE II: AUDIO SIGNAL PROFILE ---")
print(summary_table)

# =========================================================
# Visualization
# =========================================================
usages = df["usage"].astype(str).unique().tolist()
duration_data = [df.loc[df["usage"].astype(str) == usage, "duration_sec"].values for usage in usages]

plt.figure(figsize=(8, 6))
plt.boxplot(duration_data, tick_labels=usages)
plt.xticks(rotation=15)
plt.title("Duration Distribution by Protocol")
plt.ylabel("Seconds")
plt.tight_layout()
plt.show()

sr_counts = df["sampling_rate"].value_counts().sort_index()

plt.figure(figsize=(8, 6))
plt.bar(sr_counts.index.astype(str), sr_counts.values)
plt.title("Sampling Rate Consistency Check")
plt.ylabel("File Count")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# Save
df.to_csv("/kaggle/working/metadata_profiled.csv", index=False)
print("\nSaved: /kaggle/working/metadata_profiled.csv")

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.signal import resample_poly
from tqdm.auto import tqdm

# =========================
# Configuration
# =========================
TARGET_SR = 16000
WINDOW_SEC = 6.0
WINDOW_SAMPLES = int(TARGET_SR * WINDOW_SEC)
OVERLAP_RATIO = 0.10
HOP_SAMPLES = int(WINDOW_SAMPLES * (1.0 - OVERLAP_RATIO))

OUTPUT_DIR = Path("/kaggle/working/preprocessed_chunks")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

OUTPUT_CSV = Path("/kaggle/working/metadata_chunks.csv")
if OUTPUT_CSV.exists():
    OUTPUT_CSV.unlink()

df = pd.read_csv("/kaggle/working/metadata_final.csv", usecols=["path", "label", "dataset", "source_group", "usage"])

# =========================
# Fast helpers
# =========================
def load_audio_fast(path, target_sr=16000):
    audio, sr = sf.read(path, dtype="float32")

    if audio.ndim == 2:
        audio = audio.mean(axis=1)

    if sr != target_sr:
        audio = resample_poly(audio, target_sr, sr).astype(np.float32)

    return audio

def peak_normalize(x):
    peak = np.max(np.abs(x))
    if peak > 0:
        x = x / peak
    return x.astype(np.float32)

# =========================
# Processing
# =========================
print(f"Applying sliding window to {len(df)} files...")

buffer_rows = []
batch_size = 2000
total_chunks = 0

for row in tqdm(df.itertuples(index=False), total=len(df)):
    try:
        orig_path = Path(row.path)
        y = load_audio_fast(orig_path, TARGET_SR)

        total_samples = len(y)

        if total_samples <= WINDOW_SAMPLES:
            chunk = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
            chunk[:total_samples] = y
            chunk = peak_normalize(chunk)

            save_path = OUTPUT_DIR / f"{row.usage}_{row.label}_{orig_path.stem}_chk0.flac"
            sf.write(save_path, chunk, TARGET_SR)

            buffer_rows.append({
                "path": str(save_path),
                "label": int(row.label),
                "dataset": row.dataset,
                "source_group": row.source_group,
                "usage": row.usage,
                "chunk_id": 0,
            })
            total_chunks += 1

        else:
            chunk_id = 0
            for start in range(0, total_samples - WINDOW_SAMPLES + 1, HOP_SAMPLES):
                chunk = y[start:start + WINDOW_SAMPLES]
                chunk = peak_normalize(chunk)

                save_path = OUTPUT_DIR / f"{row.usage}_{row.label}_{orig_path.stem}_chk{chunk_id}.flac"
                sf.write(save_path, chunk, TARGET_SR)

                buffer_rows.append({
                    "path": str(save_path),
                    "label": int(row.label),
                    "dataset": row.dataset,
                    "source_group": row.source_group,
                    "usage": row.usage,
                    "chunk_id": chunk_id,
                })
                total_chunks += 1
                chunk_id += 1

        if len(buffer_rows) >= batch_size:
            pd.DataFrame(buffer_rows).to_csv(
                OUTPUT_CSV,
                mode="a",
                header=not OUTPUT_CSV.exists(),
                index=False
            )
            buffer_rows.clear()

    except Exception:
        continue

if buffer_rows:
    pd.DataFrame(buffer_rows).to_csv(
        OUTPUT_CSV,
        mode="a",
        header=not OUTPUT_CSV.exists(),
        index=False
    )

print("\nSliding window complete.")
print(f"Original files: {len(df)} | New chunk count: {total_chunks}")
print(f"Saved chunks to: {OUTPUT_DIR}")
print(f"Saved metadata to: {OUTPUT_CSV}")

Data Splitting

In [ ]:
import gc
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# Load chunked metadata
# =========================
df = pd.read_csv("/kaggle/working/metadata_chunks.csv")

print("Usage labels found:")
print(df["usage"].value_counts(dropna=False))

# Keep only needed columns
needed_cols = ["path", "label", "dataset", "source_group", "usage", "chunk_id"]
df = df[needed_cols].copy()

# Compact dtypes
df["label"] = df["label"].astype("int8")
for col in ["dataset", "source_group", "usage"]:
    df[col] = df[col].astype("category")

# =========================
# Rebuild original_file_id
# =========================
df["original_file_id"] = df["path"].map(lambda p: Path(p).stem.rsplit("_chk", 1)[0])

# =========================
# Normalize usage names
# =========================
usage_map = {
    "Train_SUST": "Train_Validation",
    "Test_Independent_Mozilla": "Independent_Test",
    "Test_Cross_Dataset": "Cross_Dataset_Test",
}
df["usage"] = df["usage"].astype(str).replace(usage_map)

print("\nNormalized usage labels:")
print(df["usage"].value_counts(dropna=False))

# =========================
# Define subsets
# =========================
train_pool = df.loc[df["usage"] == "Train_Validation"].copy()
test_mozilla = df.loc[df["usage"] == "Independent_Test"].copy()
test_mendeley = df.loc[df["usage"] == "Cross_Dataset_Test"].copy()

print(f"\nTotal chunks in training pool: {len(train_pool)}")

if train_pool.empty:
    raise ValueError("Training pool is empty after usage normalization. Check metadata_chunks.csv usage labels.")

# =========================
# File-level stratified split
# =========================
unique_files = train_pool[["original_file_id", "label"]].drop_duplicates()

train_files, val_files = train_test_split(
    unique_files["original_file_id"],
    test_size=0.15,
    stratify=unique_files["label"],
    random_state=42
)

train_files = set(train_files)
val_files = set(val_files)

# =========================
# Map split back to chunks
# =========================
train_df = train_pool.loc[train_pool["original_file_id"].isin(train_files)].copy()
val_df = train_pool.loc[train_pool["original_file_id"].isin(val_files)].copy()

# Cleanup
del df, train_pool, unique_files, train_files, val_files
gc.collect()

# =========================
# Summary
# =========================
print("\n--- TABLE III: FINAL DATA SPLIT (BY CHUNKS) ---")
summary_data = {
    "Subset": [
        "Training",
        "Validation",
        "Independent Test (Mozilla)",
        "Cross-Dataset Test (Mendeley)"
    ],
    "Real (0)": [
        int((train_df["label"] == 0).sum()),
        int((val_df["label"] == 0).sum()),
        int((test_mozilla["label"] == 0).sum()),
        int((test_mendeley["label"] == 0).sum()),
    ],
    "Fake (1)": [
        int((train_df["label"] == 1).sum()),
        int((val_df["label"] == 1).sum()),
        int((test_mozilla["label"] == 1).sum()),
        int((test_mendeley["label"] == 1).sum()),
    ]
}
print(pd.DataFrame(summary_data))

print("\n--- FILE-LEVEL LEAKAGE CHECK ---")
print("Train unique files:", train_df["original_file_id"].nunique())
print("Val unique files:", val_df["original_file_id"].nunique())
print("Mozilla unique files:", test_mozilla["original_file_id"].nunique())
print("Mendeley unique files:", test_mendeley["original_file_id"].nunique())
print("Train-Val overlap:", len(set(train_df['original_file_id']) & set(val_df['original_file_id'])))

# =========================
# Save outputs
# =========================
train_df.to_csv("/kaggle/working/metadata_train.csv", index=False)
val_df.to_csv("/kaggle/working/metadata_val.csv", index=False)
test_mozilla.to_csv("/kaggle/working/metadata_test_mozilla.csv", index=False)
test_mendeley.to_csv("/kaggle/working/metadata_test_mendeley.csv", index=False)

print("\nSplitting complete. Metadata saved.")

Temporal Standardization

The "Equalizer" (Padding & Trimming)

The Mathematical Balance
Let's find Mean

In [ ]:
import gc
import numpy as np
import pandas as pd
import soundfile as sf

# Load chunked metadata
df = pd.read_csv("/kaggle/working/metadata_chunks.csv", usecols=["path"])

# Representative sampling
sample_n = min(2000, len(df))
sample_df = df.sample(n=sample_n, random_state=42)

def get_duration_fast(path):
    try:
        info = sf.info(path)  # header-only read
        return info.frames / info.samplerate
    except Exception:
        return np.nan

print("Calculating Temporal Statistics...")

durations = np.array([get_duration_fast(p) for p in sample_df["path"]], dtype=np.float32)
durations = durations[~np.isnan(durations)]

if len(durations) == 0:
    print("No readable audio chunks found.")
else:
    avg_len = durations.mean()
    std_len = durations.std()
    recommended_cap = np.percentile(durations, 95)

    print("\n--- TEMPORAL STANDARDIZATION PROFILE ---")
    print(f"Average Duration: {avg_len:.2f}s")
    print(f"95th Percentile:  {recommended_cap:.2f}s")
    print(f"Standard Deviation: {std_len:.2f}s")

# Cleanup
del df, sample_df, durations
gc.collect()

# Feature Extraction

This is where we turn audio into images that the AI can actually see.we will extract two types of features to give your model "dual vision":

Mel-Spectrograms: These capture how humans hear (logarithmic scale), which is great for general voice patterns.

LFCC (Linear Frequency Cepstral Coefficients): These capture high-frequency details (linear scale), which is where Deepfake artifacts (robotic glitches) are usually hidden.


Dual-Feature Extraction (LFCC + Mel):
To capture both perceptual and architectural voice characteristics, we extracted Mel-spectrograms and Linear Frequency Cepstral Coefficients (LFCC). While Mel-spectrograms prioritize human-centric frequency distributions, LFCCs utilize a linear filter bank to preserve high-frequency spectral details, where synthetic vocoder artifacts are predominantly located.

In [ ]:
import os
import gc
import librosa
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Optimized Parameters ---
SR = 16000
MAX_SEC = 6.0
MAX_LEN = int(SR * MAX_SEC)

N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 128
N_LFCC = 40

FEATURE_PATH = "/kaggle/working/dual_features"
os.makedirs(FEATURE_PATH, exist_ok=True)

# clean old manifest if rerun
manifest_path = "/kaggle/working/metadata_features_dual.csv"
if os.path.exists(manifest_path):
    os.remove(manifest_path)

def pad_or_crop(y, max_len):
    n = len(y)

    if n < max_len:
        out = np.zeros(max_len, dtype=np.float32)
        out[:n] = y
        return out

    if n == max_len:
        return y.astype(np.float32)

    start = (n - max_len) // 2
    return y[start:start + max_len].astype(np.float32)

def peak_normalize(y):
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak
    return y.astype(np.float32)

def build_linear_filterbank(sr, n_fft, n_lfcc):
    n_freqs = n_fft // 2 + 1
    linear_bins = np.linspace(0, n_freqs - 1, n_lfcc + 2, dtype=int)
    fb = np.zeros((n_lfcc, n_freqs), dtype=np.float32)

    for i in range(n_lfcc):
        start, center, end = linear_bins[i], linear_bins[i + 1], linear_bins[i + 2]

        if center > start:
            fb[i, start:center] = np.linspace(0, 1, center - start, endpoint=False, dtype=np.float32)
        if end > center:
            fb[i, center:end] = np.linspace(1, 0, end - center, endpoint=False, dtype=np.float32)

    return fb

LINEAR_FB = build_linear_filterbank(SR, N_FFT, N_LFCC)

def extract_dual_branch(row):
    try:
        # 1. Load Audio
        y, sr = librosa.load(row["path"], sr=SR, mono=True)
        y = peak_normalize(y)
        y = pad_or_crop(y, MAX_LEN)

        # --- Branch A: Mel Spectrogram ---
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            n_mels=N_MELS
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_d = librosa.feature.delta(mel_db)
        mel_dd = librosa.feature.delta(mel_db, order=2)
        mel_final = np.stack([mel_db, mel_d, mel_dd], axis=0).astype(np.float32)

        # --- Branch B: LFCC ---
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH)) ** 2
        linear_spec = np.dot(LINEAR_FB, S)
        linear_spec = np.log(linear_spec + 1e-9).astype(np.float32)

        lfcc_base = librosa.feature.mfcc(S=linear_spec, n_mfcc=N_LFCC)
        lfcc_d = librosa.feature.delta(lfcc_base)
        lfcc_dd = librosa.feature.delta(lfcc_base, order=2)
        lfcc_final = np.stack([lfcc_base, lfcc_d, lfcc_dd], axis=0).astype(np.float32)

        # 2. Save features
        usage_dir = str(row["usage"]).lower()
        mel_dir = os.path.join(FEATURE_PATH, usage_dir, "mel")
        lfcc_dir = os.path.join(FEATURE_PATH, usage_dir, "lfcc")
        os.makedirs(mel_dir, exist_ok=True)
        os.makedirs(lfcc_dir, exist_ok=True)

        base_name = f"{row['usage']}_{row['label']}_{os.path.splitext(os.path.basename(row['path']))[0]}"
        m_path = os.path.join(mel_dir, f"{base_name}.npy")
        l_path = os.path.join(lfcc_dir, f"{base_name}.npy")

        np.save(m_path, mel_final)
        np.save(l_path, lfcc_final)

        return {
            "mel_path": m_path,
            "lfcc_path": l_path,
            "label": int(row["label"]),
            "usage": row["usage"],
            "path": row["path"]
        }

    except Exception:
        return None

# --- Sequential Extraction: safer for Kaggle RAM ---
buffer = []
header_written = False

for subset in ["train", "val", "test_mozilla", "test_mendeley"]:
    m_file = f"/kaggle/working/metadata_{subset}.csv"

    if os.path.exists(m_file):
        subset_df = pd.read_csv(m_file)
        print(f"Extracting features for {subset}...")

        for row in tqdm(subset_df.to_dict("records"), total=len(subset_df)):
            result = extract_dual_branch(row)

            if result is not None:
                buffer.append(result)

            # write in small batches
            if len(buffer) >= 500:
                pd.DataFrame(buffer).to_csv(
                    manifest_path,
                    mode="a",
                    index=False,
                    header=not header_written
                )
                header_written = True
                buffer.clear()
                gc.collect()

        del subset_df
        gc.collect()

# flush remaining rows
if buffer:
    pd.DataFrame(buffer).to_csv(
        manifest_path,
        mode="a",
        index=False,
        header=not header_written
    )
    header_written = True
    buffer.clear()

gc.collect()
print("Dual-Branch Feature Extraction Complete.")
print(f"Saved manifest: {manifest_path}")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load manifest
manifest_path = "/kaggle/working/metadata_features_dual.csv"
df = pd.read_csv(manifest_path)

# 2. Load first sample safely
try:
    mel = np.load(df["mel_path"].iloc[0], allow_pickle=False)
    lfcc = np.load(df["lfcc_path"].iloc[0], allow_pickle=False)

    print("--- FEATURE DIMENSIONS ---")
    print(f"Mel shape:  {mel.shape}  (Channels, Mels, Time)")
    print(f"LFCC shape: {lfcc.shape} (Channels, Coeffs, Time)")

    print("\n--- MATHEMATICAL RANGES ---")
    print(f"Mel Range:  {mel.min():.2f} to {mel.max():.2f}")
    print(f"LFCC Range: {lfcc.min():.2f} to {lfcc.max():.2f}")

    # 3. NaN check
    if np.isnan(mel).any() or np.isnan(lfcc).any():
        print("\nWARNING: NaNs detected in features! Check extraction.")
    else:
        print("\nFeatures are clean and ready for the model.")

except Exception as e:
    print(f"Error loading samples: {e}")
    print("Available columns in CSV:", df.columns.tolist())

In [ ]:
files_to_delete = [
    "/kaggle/working/metadata_chunks.csv",
    "/kaggle/working/master_metadata.csv",
    "/kaggle/working/metadata_profiled.csv",
    "/kaggle/working/metadata_final.csv"
]

for f in files_to_delete:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

print("Cleanup complete.")

In [ ]:
df_master = pd.read_csv("/kaggle/working/metadata_features_dual.csv")

print("--- DIAGNOSTIC ---")
print("Columns found:", df_master.columns.tolist())

if "usage" in df_master.columns:
    print("\nUnique values in 'usage':")
    print(df_master["usage"].value_counts())
else:
    print("\nERROR: 'usage' column is missing!")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load master feature file
df_master = pd.read_csv("/kaggle/working/metadata_features_dual.csv")

print("--- RESTRUCTURING DATASET FOR TRAINING ---")

# 🔧 Normalize usage names (IMPORTANT)
usage_map = {
    "Train_SUST": "Train_Validation",
    "Test_Independent_Mozilla": "Independent_Test",
    "Test_Cross_Dataset": "Cross_Dataset_Test"
}
df_master["usage"] = df_master["usage"].astype(str).replace(usage_map)

print("\nUsage distribution:")
print(df_master["usage"].value_counts())

# =========================================================
# 1. Training/Validation (SUST)
# =========================================================
sust_df = df_master[df_master["usage"] == "Train_Validation"].copy()

if sust_df.empty:
    raise ValueError("Training data is empty! Check usage labels.")

train_df, val_df = train_test_split(
    sust_df,
    test_size=0.15,
    stratify=sust_df["label"],
    random_state=42
)

# =========================================================
# 2. Independent Test Sets
# =========================================================
test_mozilla = df_master[df_master["usage"] == "Independent_Test"].copy()
test_mendeley = df_master[df_master["usage"] == "Cross_Dataset_Test"].copy()

# =========================================================
# 3. Save
# =========================================================
train_df.to_csv("/kaggle/working/metadata_dual_train.csv", index=False)
val_df.to_csv("/kaggle/working/metadata_dual_val.csv", index=False)
test_mozilla.to_csv("/kaggle/working/metadata_dual_test_mozilla.csv", index=False)
test_mendeley.to_csv("/kaggle/working/metadata_dual_test_mendeley.csv", index=False)

# =========================================================
# Summary
# =========================================================
print("\n--- FINAL SPLIT SUMMARY ---")
print(f"Training Set: {len(train_df)} chunks")
print(f"Validation Set: {len(val_df)} chunks")
print(f"Mozilla Test: {len(test_mozilla)} chunks")
print(f"Mendeley Test: {len(test_mendeley)} chunks")

Dual-Branch DataLoader

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class BengaliDeepfakeDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Safe loading
        mel = np.load(row["mel_path"], allow_pickle=False)
        lfcc = np.load(row["lfcc_path"], allow_pickle=False)

        mel = torch.from_numpy(mel).float()
        lfcc = torch.from_numpy(lfcc).float()

        # IMPORTANT: use long for classification
        label = torch.tensor(int(row["label"]), dtype=torch.long)

        return mel, lfcc, label

# =========================================================
# DataLoaders (optimized for Kaggle)
# =========================================================
BATCH_SIZE = 32

train_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_train.csv"),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,        # small for Kaggle
    pin_memory=True
)

val_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_val.csv"),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoaders Active. Train batches: {len(train_loader)}")

The Data Audit (Leakage & Balance Check)

In [ ]:
# Load training metadata
train_df = pd.read_csv("/kaggle/working/metadata_dual_train.csv")

print("--- TABLE I: DATASET AUDIT ---")

# 1. Class balance
balance = train_df["label"].value_counts().sort_index()
real_count = int(balance.get(0, 0))
fake_count = int(balance.get(1, 0))
print(f"Class Balance: Real={real_count}, Fake={fake_count}")

# 2. Leakage / group integrity check
# Recover original source id from saved feature name
train_df["orig_id"] = train_df["mel_path"].apply(
    lambda x: os.path.basename(x).rsplit("_chk", 1)[0]
)

unique_ids = train_df["orig_id"].nunique()
total_chunks = len(train_df)
avg_chunks = total_chunks / unique_ids if unique_ids > 0 else 0.0

print(f"Total Chunks: {total_chunks}")
print(f"Unique Original Sources: {unique_ids}")
print(f"Average chunks per source: {avg_chunks:.2f}")

if total_chunks > unique_ids:
    print("Group Integrity Verified: Multiple chunks exist per source, but are isolated in Train.")
else:
    print("No multi-chunk grouping detected.")

In [ ]:
train_df = pd.read_csv("/kaggle/working/metadata_dual_train.csv")
val_df = pd.read_csv("/kaggle/working/metadata_dual_val.csv")

def get_id(x):
    return os.path.basename(x).rsplit("_chk", 1)[0]

train_ids = set(train_df["mel_path"].apply(get_id))
val_ids = set(val_df["mel_path"].apply(get_id))

overlap = train_ids & val_ids

print(f"Train-Val overlap: {len(overlap)}")

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Load feature manifest
df = pd.read_csv("/kaggle/working/metadata_features_dual.csv")

print("Available usage values:")
print(df["usage"].value_counts(dropna=False))

# Recover original file id
df["orig_id"] = df["mel_path"].apply(
    lambda x: os.path.basename(x).rsplit("_chk", 1)[0]
)

# Normalize usage names
usage_map = {
    "Train_SUST": "Train_Validation",
    "Test_Independent_Mozilla": "Independent_Test",
    "Test_Cross_Dataset": "Cross_Dataset_Test",
}
df["usage"] = df["usage"].astype(str).replace(usage_map)

print("\nNormalized usage values:")
print(df["usage"].value_counts(dropna=False))

# Training subset
sust_df = df[df["usage"] == "Train_Validation"].copy()

if sust_df.empty:
    raise ValueError("Train_Validation subset is empty. Check printed usage values above.")

# Unique original files for leakage-safe split
unique_files = sust_df[["orig_id", "label"]].drop_duplicates()

train_ids, val_ids = train_test_split(
    unique_files["orig_id"],
    test_size=0.15,
    stratify=unique_files["label"],
    random_state=42
)

train_ids = set(train_ids)
val_ids = set(val_ids)

# Map back to all chunks/features
train_df = sust_df[sust_df["orig_id"].isin(train_ids)].copy()
val_df = sust_df[sust_df["orig_id"].isin(val_ids)].copy()

# Test sets
test_mozilla = df[df["usage"] == "Independent_Test"].copy()
test_mendeley = df[df["usage"] == "Cross_Dataset_Test"].copy()

# Save corrected splits
train_df.to_csv("/kaggle/working/metadata_dual_train.csv", index=False)
val_df.to_csv("/kaggle/working/metadata_dual_val.csv", index=False)
test_mozilla.to_csv("/kaggle/working/metadata_dual_test_mozilla.csv", index=False)
test_mendeley.to_csv("/kaggle/working/metadata_dual_test_mendeley.csv", index=False)

print("\nFixed splits saved.")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Mozilla:", len(test_mozilla))
print("Mendeley:", len(test_mendeley))

# Leakage check
train_orig = set(train_df["orig_id"])
val_orig = set(val_df["orig_id"])
print("Train-Val overlap:", len(train_orig & val_orig))

Feature Normalization.

In [ ]:

def compute_global_stats(file_list, n_samples=5000):
    """
    Memory-friendly approximate global mean/std.
    Uses random sampling + running aggregation.
    """

    sample_files = np.random.choice(
        file_list,
        min(n_samples, len(file_list)),
        replace=False
    )

    total_sum = 0.0
    total_sq_sum = 0.0
    total_count = 0

    for f in tqdm(sample_files, desc="Calculating Stats"):
        data = np.load(f, allow_pickle=False)  # safer

        total_sum += data.sum()
        total_sq_sum += (data ** 2).sum()
        total_count += data.size

    mean = total_sum / total_count
    std = np.sqrt((total_sq_sum / total_count) - (mean ** 2))

    return mean, std


print("Calculating normalization constants (Training partition only)...")

mel_mean, mel_std = compute_global_stats(train_df["mel_path"].values)
lfcc_mean, lfcc_std = compute_global_stats(train_df["lfcc_path"].values)

print("\n--- NORMALIZATION CONSTANTS ---")
print(f"Mel:  Mean={mel_mean:.4f}, Std={mel_std:.4f}")
print(f"LFCC: Mean={lfcc_mean:.4f}, Std={lfcc_std:.4f}")

Integrating Normalization into the DataLoader

In [ ]:
class BengaliDeepfakeDataset(Dataset):
    def __init__(self, csv_path, mel_stats, lfcc_stats):
        self.df = pd.read_csv(csv_path)
        self.m_mean, self.m_std = mel_stats
        self.l_mean, self.l_std = lfcc_stats
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load
        mel = np.load(row['mel_path'], allow_pickle=False)
        lfcc = np.load(row['lfcc_path'], allow_pickle=False)
        
        # Normalize
        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)
        
        return (
            torch.from_numpy(mel).float(),
            torch.from_numpy(lfcc).float(),
            torch.tensor(int(row['label']), dtype=torch.long)  # FIXED
        )

# =========================
# Initialize Loaders
# =========================
mel_stats = (mel_mean, mel_std)
lfcc_stats = (lfcc_mean, lfcc_std)

train_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_train.csv", mel_stats, lfcc_stats),
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_val.csv", mel_stats, lfcc_stats),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nNormalization applied. Model is now mathematically balanced.")

Finalizing Normalization Stats

In [ ]:
print("Calculating normalization constants (Training partition only)...")

mel_mean, mel_std = compute_global_stats(train_df["mel_path"].values)
lfcc_mean, lfcc_std = compute_global_stats(train_df["lfcc_path"].values)

print("\n--- NORMALIZATION CONSTANTS (FOR JOURNAL) ---")
print(f"Mel:  Mean={mel_mean:.4f}, Std={mel_std:.4f}")
print(f"LFCC: Mean={lfcc_mean:.4f}, Std={lfcc_std:.4f}")

# Optional: save for reproducibility
with open("/kaggle/working/normalization_stats.txt", "w") as f:
    f.write(f"Mel Mean: {mel_mean}\n")
    f.write(f"Mel Std: {mel_std}\n")
    f.write(f"LFCC Mean: {lfcc_mean}\n")
    f.write(f"LFCC Std: {lfcc_std}\n")

print("\nSaved normalization constants to file.")

In [ ]:
# Get one batch from the loader
test_mel, test_lfcc, _ = next(iter(train_loader))

print("--- NORMALIZATION CHECK ---")
print(f"Normalized Mel Mean:  {test_mel.mean().item():.4f} (Target: ~0)")
print(f"Normalized Mel Std:   {test_mel.std().item():.4f} (Target: ~1)")
print(f"Normalized LFCC Mean: {test_lfcc.mean().item():.4f} (Target: ~0)")
print(f"Normalized LFCC Std:  {test_lfcc.std().item():.4f} (Target: ~1)")

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class BengaliDeepfakeDataset(Dataset):
    def __init__(self, csv_path, mel_stats, lfcc_stats):
        self.df = pd.read_csv(csv_path)
        self.m_mean, self.m_std = mel_stats
        self.l_mean, self.l_std = lfcc_stats
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and apply Z-score normalization
        mel = np.load(row["mel_path"], allow_pickle=False)
        lfcc = np.load(row["lfcc_path"], allow_pickle=False)

        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)
        
        return (
            torch.from_numpy(mel).float(),
            torch.from_numpy(lfcc).float(),
            torch.tensor(int(row["label"]), dtype=torch.long)
        )

# Initialize with calculated constants
mel_stats = (mel_mean, mel_std)
lfcc_stats = (lfcc_mean, lfcc_std)

train_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_train.csv", mel_stats, lfcc_stats),
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_val.csv", mel_stats, lfcc_stats),
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nFeatures normalized. DataLoaders are ready for training.")

Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def run_scientific_eda(loader):
    # Grab one batch for analysis
    mels, lfccs, labels = next(iter(loader))
    
    # Separate Real (0) and Fake (1)
    real_idx = (labels == 0).nonzero(as_tuple=True)[0]
    fake_idx = (labels == 1).nonzero(as_tuple=True)[0]

    plt.figure(figsize=(16, 10))
    plt.suptitle("Bengali Deepfake Forensic Analysis: Mel vs LFCC", fontsize=18, fontweight='bold')

    # --- 1. MEL SPECTROGRAM COMPARISON ---
    # Real Mel (Average over the batch)
    plt.subplot(2, 2, 1)
    avg_real_mel = mels[real_idx].mean(dim=(0, 1)).cpu().numpy()
    plt.imshow(avg_real_mel, aspect='auto', origin='lower', cmap='magma')
    plt.title("A. Average REAL Mel (Perceptual Pattern)", fontsize=13)
    plt.colorbar(label='Normalized Power')

    # Fake Mel (Average over the batch)
    plt.subplot(2, 2, 2)
    avg_fake_mel = mels[fake_idx].mean(dim=(0, 1)).cpu().numpy()
    plt.imshow(avg_fake_mel, aspect='auto', origin='lower', cmap='magma')
    plt.title("B. Average FAKE Mel (Artifact Detection)", fontsize=13)
    plt.colorbar(label='Normalized Power')

    # --- 2. DISTRIBUTION ANALYIS (The Statistical Gap) ---
    plt.subplot(2, 1, 2)
    sns.kdeplot(mels[real_idx].flatten().numpy(), label="Real (Mel)", color="blue", fill=True, alpha=0.3)
    sns.kdeplot(mels[fake_idx].flatten().numpy(), label="Fake (Mel)", color="red", fill=True, alpha=0.3)
    sns.kdeplot(lfccs[real_idx].flatten().numpy(), label="Real (LFCC)", color="green", ls="--")
    sns.kdeplot(lfccs[fake_idx].flatten().numpy(), label="Fake (LFCC)", color="orange", ls="--")
    
    plt.title("C. Feature Distribution: Real vs Fake Bengali Audio", fontsize=13)
    plt.xlabel("Normalized Value (Z-Score)")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.2)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Print Numerical Insights for the Paper
    print(f"--- EDA INSIGHTS ---")
    print(f"Real Mel Variance: {mels[real_idx].var().item():.4f}")
    print(f"Fake Mel Variance: {mels[fake_idx].var().item():.4f}")
    print(f"Statistical Difference (Mean): {abs(mels[real_idx].mean() - mels[fake_idx].mean()):.4f}")

# Execute EDA
run_scientific_eda(train_loader)

In [ ]:

def plot_forensic_comparison(loader):
    # Grab a batch and find one Real and one Fake sample
    mels, _, labels = next(iter(loader))
    real_idx = (labels == 0).nonzero()[0][0].item()
    fake_idx = (labels == 1).nonzero()[0][0].item()

    fig, axes = plt.subplots(2, 2, figsize=(18, 10))
    
    # 1. Real Spectrogram (Static Channel)
    im1 = axes[0, 0].imshow(mels[real_idx, 0].cpu().numpy(), aspect='auto', origin='lower', cmap='viridis')
    axes[0, 0].set_title("Authentic Bengali Spectrogram (Mel)", fontsize=14)
    plt.colorbar(im1, ax=axes[0, 0])

    # 2. Fake Spectrogram (Static Channel)
    im2 = axes[0, 1].imshow(mels[fake_idx, 0].cpu().numpy(), aspect='auto', origin='lower', cmap='magma')
    axes[0, 1].set_title("Deepfake Bengali Spectrogram (Mel)", fontsize=14)
    plt.colorbar(im2, ax=axes[0, 1])

    # 3. Distribution Comparison (Instead of Raw Waveform)
    # Note: Since we only have Mel/LFCC in the loader, we visualize the 
    # 'energy' distribution which mimics the waveform envelope.
    axes[1, 0].plot(mels[real_idx, 0].mean(dim=0).cpu().numpy(), label='Real', color='blue', alpha=0.8)
    axes[1, 0].plot(mels[fake_idx, 0].mean(dim=0).cpu().numpy(), label='Fake', color='red', alpha=0.6)
    axes[1, 0].set_title("Temporal Energy Envelope (Real vs Fake)", fontsize=14)
    axes[1, 0].legend()

    # 4. Feature Variance Map
    diff = np.abs(mels[real_idx, 0].cpu().numpy() - mels[fake_idx, 0].cpu().numpy())
    im3 = axes[1, 1].imshow(diff, aspect='auto', origin='lower', cmap='hot')
    axes[1, 1].set_title("Spectral Delta (Magnitude of Difference)", fontsize=14)
    plt.colorbar(im3, ax=axes[1, 1])

    plt.tight_layout()
    plt.show()

plot_forensic_comparison(train_loader)

T-SNE

In [ ]:
from sklearn.manifold import TSNE

def run_tsne_analysis(loader, n_components=2, n_samples=1000):
    model_input = []
    model_labels = []
    
    # Collect a subset of data
    for i, (mel, lfcc, label) in enumerate(loader):
        # Flatten the features (3 channels * Freq * Time)
        flat_features = mel.view(mel.size(0), -1).numpy()
        model_input.append(flat_features)
        model_labels.append(label.numpy())
        if i * loader.batch_size >= n_samples: break

    X = np.concatenate(model_input)
    y = np.concatenate(model_labels)

    # Compute t-SNE
    print("Computing t-SNE (this may take a minute)...")
    tsne = TSNE(n_components=n_components, perplexity=30, random_state=42)
    X_embedded = tsne.fit_transform(X)

    # Plot
    plt.figure(figsize=(10, 7))
    df_tsne = pd.DataFrame({
        'x': X_embedded[:, 0],
        'y': X_embedded[:, 1],
        'Label': ['Real' if l == 0 else 'Fake' for l in y]
    })
    
    sns.scatterplot(data=df_tsne, x='x', y='y', hue='Label', palette=['blue', 'red'], alpha=0.6)
    plt.title("T-SNE Visualization of Bengali Audio Features", fontsize=15)
    plt.xlabel("TSNE Dimension 1")
    plt.ylabel("TSNE Dimension 2")
    plt.legend(bbox_to_anchor=(1.05, 1), loc=2)
    plt.grid(True, alpha=0.3)
    plt.show()

run_tsne_analysis(train_loader)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_residual_spectral_analysis(loader):
    mels, _, labels = next(iter(loader))
    
    # Get all real and all fake in this batch
    real_mels = mels[labels == 0, 0, :, :].mean(0).cpu().numpy()
    fake_mels = mels[labels == 1, 0, :, :].mean(0).cpu().numpy()
    
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Plot Real
    im1 = axes[0].imshow(real_mels, aspect='auto', origin='lower', cmap='viridis')
    axes[0].set_title("Mean Real Spectrogram", fontsize=12)
    plt.colorbar(im1, ax=axes[0])
    
    # Plot Fake
    im2 = axes[1].imshow(fake_mels, aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title("Mean Fake Spectrogram", fontsize=12)
    plt.colorbar(im2, ax=axes[1])
    
    # Plot the RESIDUAL (Difference)
    # This is what the AI looks for!
    residual = np.abs(real_mels - fake_mels)
    im3 = axes[2].imshow(residual, aspect='auto', origin='lower', cmap='inferno')
    axes[2].set_title("Forensic Residual (The 'Tell')", fontsize=12)
    plt.colorbar(im3, ax=axes[2])
    
    plt.show()

plot_residual_spectral_analysis(train_loader)

In [ ]:

def run_energy_audit_fixed(loader):
    real_energies = []
    fake_energies = []
    
    # We will look at 50 batches maximum to be sure we see both classes
    max_batches = 50
    
    for i, (mel, _, labels) in enumerate(loader):
        # Calculate energy for each sample in the batch
        # shape of mel is (Batch, 3, Freq, Time)
        energy = mel.mean(dim=(1, 2, 3)).cpu().numpy() 
        
        # Convert labels to numpy for indexing
        lbls = labels.cpu().numpy().astype(int)
        
        real_energies.extend(energy[lbls == 0])
        fake_energies.extend(energy[lbls == 1])
        
        if i >= max_batches:
            break

    if len(real_energies) == 0 or len(fake_energies) == 0:
        print("⚠️ Warning: One of the classes was not found in the first 50 batches.")
        return

    plt.figure(figsize=(10, 6))
    sns.kdeplot(real_energies, color="blue", label="Real", fill=True, alpha=0.4)
    sns.kdeplot(fake_energies, color="red", label="Fake", fill=True, alpha=0.4)
    
    plt.title("Energy Distribution Audit: Real vs Fake Bengali Audio", fontsize=14)
    plt.xlabel("Mean Spectral Energy (Normalized)")
    plt.ylabel("Density")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(f"--- Energy Statistics ---")
    print(f"Real Energy Mean: {np.mean(real_energies):.4f}")
    print(f"Fake Energy Mean: {np.mean(fake_energies):.4f}")
    print(f"Energy Shift: {abs(np.mean(real_energies) - np.mean(fake_energies)):.4f}")

# Run it
run_energy_audit_fixed(train_loader)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_spectral_envelope(df):
    # Grab one real and one fake sample from the paths in your DF
    real_path = df[df['label'] == 0]['mel_path'].iloc[0]
    fake_path = df[df['label'] == 1]['mel_path'].iloc[0]
    
    plt.figure(figsize=(16, 6))
    
    for i, (path, title, color) in enumerate([(real_path, "Real", "blue"), (fake_path, "Fake", "red")]):
        # Load the saved Mel feature (Shape: 3, Freq, Time)
        mel = np.load(path)
        
        # Collapse frequency and channels to get temporal energy (The "Waveform" look)
        energy = mel[0].mean(axis=0) 
        
        plt.subplot(1, 2, i+1)
        plt.fill_between(range(len(energy)), energy, color=color, alpha=0.3)
        plt.plot(energy, color=color, lw=1.5)
        plt.title(f"Bengali {title} Temporal Energy Envelope", fontsize=14)
        plt.xlabel("Time Frames")
        plt.ylabel("Normalized Magnitude")
        plt.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

# Run this - it uses the files we know exist!
plot_spectral_envelope(train_df)

In [ ]:
import pandas as pd

def final_data_audit(train_csv, val_csv, mel_mean, mel_std, lfcc_mean, lfcc_std):
    # Load the CSVs used by your loaders
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    
    # Calculate counts
    train_real = int((train_df["label"] == 0).sum())
    train_fake = int((train_df["label"] == 1).sum())
    val_real = int((val_df["label"] == 0).sum())
    val_fake = int((val_df["label"] == 1).sum())

    train_total = len(train_df)
    val_total = len(val_df)

    train_fake_ratio = train_fake / train_total if train_total > 0 else 0.0
    val_fake_ratio = val_fake / val_total if val_total > 0 else 0.0
    
    print("--- FINAL VERIFIED DATA COUNTS ---")
    print(f"{'Partition':<15} | {'Total':<8} | {'Real (0)':<10} | {'Fake (1)':<10} | {'Fake Ratio'}")
    print("-" * 70)
    print(f"{'Training':<15} | {train_total:<8} | {train_real:<10} | {train_fake:<10} | {train_fake_ratio:.2%}")
    print(f"{'Validation':<15} | {val_total:<8} | {val_real:<10} | {val_fake:<10} | {val_fake_ratio:.2%}")
    print("-" * 70)
    
    # Normalization constants
    print("\nNormalization Constants Locked:")
    print(f"  Mel:  mu={mel_mean:.4f}, sigma={mel_std:.4f}")
    print(f"  LFCC: mu={lfcc_mean:.4f}, sigma={lfcc_std:.4f}")
    
    if abs(0.5 - train_fake_ratio) < 0.05:
        print("\nClass balance is excellent. No sampling bias detected.")
    else:
        print("\nClass imbalance detected. Consider weighted loss if needed.")

# Execute audit
final_data_audit(
    "/kaggle/working/metadata_dual_train.csv",
    "/kaggle/working/metadata_dual_val.csv",
    mel_mean, mel_std,
    lfcc_mean, lfcc_std
)

In [ ]:
# Final Batch Audit
train_batches = len(train_loader)
val_batches = len(val_loader)
batch_size = train_loader.batch_size

# Extract one sample batch
sample_mel, sample_lfcc, sample_labels = next(iter(train_loader))

print("--- FINAL LOADER AUDIT ---")
print(f"Total Train Batches: {train_batches}")
print(f"Total Val Batches:   {val_batches}")
print(f"Batch Size:          {batch_size}")
print("-" * 30)
print(f"Mel Input Shape:     {tuple(sample_mel.shape)}  (Target: [Batch, 3, Freq, Time])")
print(f"LFCC Input Shape:    {tuple(sample_lfcc.shape)} (Target: [Batch, 3, Freq, Time])")
print(f"Labels Shape:        {tuple(sample_labels.shape)}")
print(f"Labels Dtype:        {sample_labels.dtype}")
print("-" * 30)

# Batch composition
real_count = int((sample_labels == 0).sum().item())
fake_count = int((sample_labels == 1).sum().item())
print(f"Batch 1 Composition: {real_count} Real, {fake_count} Fake")

# NaN check
print("-" * 30)
print(f"Mel has NaN:         {torch.isnan(sample_mel).any().item()}")
print(f"LFCC has NaN:        {torch.isnan(sample_lfcc).any().item()}")

if train_batches > 0 and val_batches > 0:
    print("\nLoaders are fully primed for training.")

model pipeline.

In [ ]:

import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# BSCAN-v2: Residual + SE Attention
# ==========================================

class SEModule(nn.Module):
    """Squeeze-and-Excitation Module for channel-wise attention."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class SEResBlock(nn.Module):
    """Residual block with Squeeze-and-Excitation attention."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.se = SEModule(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels,
                    kernel_size=1, stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)

        out = out + identity
        out = self.relu(out)
        return out


class EfficientAttention(nn.Module):
    """Selective temporal attention."""
    def __init__(self, in_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv1d(in_dim, in_dim, kernel_size=1, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv1d(in_dim, 1, kernel_size=1, bias=True)
        )

    def forward(self, x):
        # x: [B, C, T]
        w = torch.softmax(self.attn(x), dim=-1)   # [B, 1, T]
        out = (x * w).sum(dim=-1)                 # [B, C]
        return out


class ResidualBranch(nn.Module):
    """Feature extractor for one branch."""
    def __init__(self, in_channels=3, hidden_dim=64, use_attention=True):
        super().__init__()
        self.layer1 = SEResBlock(in_channels, 32, stride=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        self.layer2 = SEResBlock(32, hidden_dim, stride=1)
        self.pool2 = nn.AdaptiveAvgPool2d((1, None))

        self.use_attention = use_attention
        if self.use_attention:
            self.attn = EfficientAttention(hidden_dim)

    def forward(self, x):
        # x: [B, 3, F, T]
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2(x)
        x = self.pool2(x).squeeze(2)   # [B, C, T]

        if self.use_attention:
            x = self.attn(x)           # [B, C]
        else:
            x = x.mean(dim=-1)         # [B, C]

        return x


class BSCAN(nn.Module):
    """Dual-Branch Spectro-Cepstral Residual Attention Network."""
    def __init__(self, use_attention=True, mode="dual", hidden_dim=64, dropout=0.3):
        super().__init__()
        assert mode in ["dual", "mel_only", "lfcc_only"], "Invalid mode"

        self.mode = mode

        if mode in ["dual", "mel_only"]:
            self.mel_branch = ResidualBranch(
                in_channels=3,
                hidden_dim=hidden_dim,
                use_attention=use_attention
            )

        if mode in ["dual", "lfcc_only"]:
            self.lfcc_branch = ResidualBranch(
                in_channels=3,
                hidden_dim=hidden_dim,
                use_attention=use_attention
            )

        in_features = 0
        if mode in ["dual", "mel_only"]:
            in_features += hidden_dim
        if mode in ["dual", "lfcc_only"]:
            in_features += hidden_dim

        self.classifier = nn.Sequential(
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, mel, lfcc=None):
        features = []

        if self.mode in ["dual", "mel_only"]:
            features.append(self.mel_branch(mel))

        if self.mode in ["dual", "lfcc_only"]:
            if lfcc is None:
                raise ValueError("lfcc input is required for mode='dual' or mode='lfcc_only'")
            features.append(self.lfcc_branch(lfcc))

        fused = torch.cat(features, dim=1)
        logits = self.classifier(fused).squeeze(1)   # [B]
        return logits

In [ ]:
import torch
import gc
from torch.utils.data import DataLoader

# 1. Clear GPU Memory
torch.cuda.empty_cache()
gc.collect()

# 2. IMPORTANT: Match label type with loss (BCEWithLogitsLoss → float32)

class BengaliDeepfakeDataset(Dataset):
    def __init__(self, csv_path, mel_stats, lfcc_stats):
        self.df = pd.read_csv(csv_path)
        self.m_mean, self.m_std = mel_stats
        self.l_mean, self.l_std = lfcc_stats
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        mel = np.load(row["mel_path"], allow_pickle=False)
        lfcc = np.load(row["lfcc_path"], allow_pickle=False)

        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)

        return (
            torch.from_numpy(mel).float(),
            torch.from_numpy(lfcc).float(),
            torch.tensor(float(row["label"]), dtype=torch.float32)  # FIXED
        )

# 3. Reinitialize loaders (smaller batch for stability)
BATCH_SIZE = 16

train_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_train.csv", mel_stats, lfcc_stats),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    BengaliDeepfakeDataset("/kaggle/working/metadata_dual_val.csv", mel_stats, lfcc_stats),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Loaders reset. Batch size: {BATCH_SIZE}. GPU cache cleared.")

In [ ]:
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

# =========================
# Configuration
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCHS = 15
PATIENCE = 5

best_score = -1.0
trigger_times = 0
best_threshold = 0.5

model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(DEVICE)

optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

# Better for improving val metrics than fixed cosine here
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
    "val_acc": [],
    "val_score": [],
    "best_threshold": []
}

print(f"BSCAN Training Phase Started (Device: {DEVICE})")

for epoch in range(EPOCHS):
    # =========================
    # Training
    # =========================
    model.train()
    train_loss = 0.0
    all_train_probs = []
    all_train_labels = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{EPOCHS} [Train]")

    for mels, lfccs, labels in progress:
        mels = mels.to(DEVICE, non_blocking=True)
        lfccs = lfccs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True).view(-1)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            outputs = model(mels, lfccs).view(-1)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()

        probs = torch.sigmoid(outputs)
        all_train_probs.extend(probs.detach().cpu().numpy().tolist())
        all_train_labels.extend(labels.detach().cpu().numpy().astype(np.int32).tolist())

        progress.set_postfix(loss=f"{train_loss / max(1, progress.n):.4f}")

    # train metrics at fixed threshold 0.5
    train_preds = (np.array(all_train_probs) > 0.5).astype(np.int32)
    train_f1 = f1_score(all_train_labels, train_preds, average="macro")

    # =========================
    # Validation
    # =========================
    model.eval()
    val_loss = 0.0
    all_val_probs = []
    all_val_labels = []

    with torch.no_grad():
        for mels, lfccs, labels in val_loader:
            mels = mels.to(DEVICE, non_blocking=True)
            lfccs = lfccs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True).view(-1)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                outputs = model(mels, lfccs).view(-1)
                loss = criterion(outputs, labels)

            val_loss += loss.item()

            probs = torch.sigmoid(outputs)
            all_val_probs.extend(probs.cpu().numpy().tolist())
            all_val_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    # =========================
    # Threshold search on validation
    # =========================
    val_probs_np = np.array(all_val_probs)
    val_labels_np = np.array(all_val_labels)

    threshold_candidates = np.arange(0.30, 0.71, 0.02)

    best_epoch_f1 = -1.0
    best_epoch_acc = -1.0
    best_epoch_score = -1.0
    best_epoch_threshold = 0.5

    for th in threshold_candidates:
        preds = (val_probs_np > th).astype(np.int32)
        f1 = f1_score(val_labels_np, preds, average="macro")
        acc = accuracy_score(val_labels_np, preds) * 100.0

        # combined objective: prioritize F1, but include accuracy
        score = 0.7 * f1 + 0.3 * (acc / 100.0)

        if score > best_epoch_score:
            best_epoch_score = score
            best_epoch_f1 = f1
            best_epoch_acc = acc
            best_epoch_threshold = float(th)

    best_threshold = best_epoch_threshold

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_f1"].append(train_f1)
    history["val_f1"].append(best_epoch_f1)
    history["val_acc"].append(best_epoch_acc)
    history["val_score"].append(best_epoch_score)
    history["best_threshold"].append(best_epoch_threshold)

    scheduler.step(best_epoch_score)

    print(f"\nEpoch {epoch + 1} Results:")
    print(f"Train Loss: {avg_train_loss:.4f} | Train F1: {train_f1:.4f}")
    print(f"Val Loss:   {avg_val_loss:.4f} | Val F1: {best_epoch_f1:.4f} | Val Acc: {best_epoch_acc:.2f}% | Th: {best_epoch_threshold:.2f}")

    # =========================
    # Save checkpoint + Early stopping
    # =========================
    if best_epoch_score > best_score:
        best_score = best_epoch_score
        trigger_times = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_f1": best_epoch_f1,
                "val_acc": best_epoch_acc,
                "val_score": best_epoch_score,
                "best_threshold": best_epoch_threshold,
                "history": history,
            },
            "/kaggle/working/BSCAN_Best_Model.pth"
        )
        print("Success: combined validation score improved. Model saved.")
    else:
        trigger_times += 1
        print(f"No improvement. Early-stop counter: {trigger_times}/{PATIENCE}")

        if trigger_times >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Save training logs ---
df_history = pd.DataFrame(history)
df_history.to_csv("/kaggle/working/BSCAN_Journal_Training_Logs.csv", index=False)
print("Training logs archived: /kaggle/working/BSCAN_Journal_Training_Logs.csv")

# --- 2. Professional plotting setup ---
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14
})

epochs_range = np.arange(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
ax1, ax2 = axes[0]
ax3, ax4 = axes[1]

# A. Loss Curves
ax1.plot(
    epochs_range, history["train_loss"],
    label="Training", lw=2, marker="o", markersize=3, alpha=0.85
)
ax1.plot(
    epochs_range, history["val_loss"],
    label="Validation", lw=2, marker="x", markersize=4
)
ax1.set_title("(a) Convergence Analysis (Loss)", pad=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss (BCE)")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(frameon=True)

# B. Accuracy Curve
ax2.plot(
    epochs_range, history["val_acc"],
    label="Val. Accuracy", lw=2, marker="s", markersize=3
)
ax2.set_title("(b) Classification Performance", pad=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(frameon=True)

# C. F1 Score
ax3.plot(
    epochs_range, history["val_f1"],
    label="Val. Macro-F1", lw=2, marker="^", markersize=4
)
if "train_f1" in history:
    ax3.plot(
        epochs_range, history["train_f1"],
        label="Train Macro-F1", lw=1.8, marker="o", markersize=3, alpha=0.8
    )
ax3.set_title("(c) Generalization (Macro-F1)", pad=12)
ax3.set_xlabel("Epoch")
ax3.set_ylabel("F1 Score")
ax3.set_ylim(0, 1.05)
ax3.grid(True, linestyle="--", alpha=0.4)
ax3.legend(frameon=True)

# D. Combined score / threshold
if "val_score" in history:
    ax4.plot(
        epochs_range, history["val_score"],
        label="Val. Combined Score", lw=2, marker="d", markersize=3
    )
if "best_threshold" in history:
    ax4.plot(
        epochs_range, history["best_threshold"],
        label="Best Threshold", lw=1.8, marker="x", markersize=4, alpha=0.85
    )
ax4.set_title("(d) Validation Selection Signals", pad=12)
ax4.set_xlabel("Epoch")
ax4.set_ylabel("Score / Threshold")
ax4.grid(True, linestyle="--", alpha=0.4)
ax4.legend(frameon=True)

plt.tight_layout()

# --- 3. Export ---
plt.savefig("/kaggle/working/BSCAN_Full_Performance_Report.png", dpi=300, bbox_inches="tight")
plt.savefig("/kaggle/working/BSCAN_Figure_1.pdf", bbox_inches="tight")

print("Saved:")
print("/kaggle/working/BSCAN_Journal_Training_Logs.csv")
print("/kaggle/working/BSCAN_Full_Performance_Report.png")
print("/kaggle/working/BSCAN_Figure_1.pdf")

plt.show()

In [ ]:
# Load Mozilla test directly
test_df = pd.read_csv("/kaggle/working/metadata_dual_test_mozilla.csv")

plot_original_dataset_metrics(
    "/kaggle/working/BSCAN_Best_Model.pth",
    {"Mozilla Independent": test_df},  # create dict on the fly
    mel_stats,
    lfcc_stats
)

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# =========================
# DATASET (DIRECT FROM DF)
# =========================
class BengaliDeepfakeDataset_Direct(Dataset):
    def __init__(self, df, mel_stats, lfcc_stats):
        self.df = df
        self.m_mean, self.m_std = mel_stats
        self.l_mean, self.l_std = lfcc_stats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        mel = np.load(row["mel_path"], allow_pickle=False)
        lfcc = np.load(row["lfcc_path"], allow_pickle=False)

        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)

        return (
            torch.from_numpy(mel).float(),
            torch.from_numpy(lfcc).float(),
            torch.tensor(float(row["label"]), dtype=torch.float32)
        )

# =========================
# LOAD TEST DATA
# =========================
test_df = pd.read_csv("/kaggle/working/metadata_dual_test_mozilla.csv")

# =========================
# RUN EVALUATION
# =========================
plot_original_dataset_metrics(
    "/kaggle/working/BSCAN_Best_Model.pth",
    {"Mozilla Independent": test_df},
    mel_stats,
    lfcc_stats
)

Multi-Dataset Evaluation

In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from IPython.display import display

# =========================================================
# 1. DATASET CLASS
# =========================================================
class BengaliDeepfakeDataset_Direct(Dataset):
    def __init__(self, df, m_stats, l_stats):
        self.df = df.reset_index(drop=True)
        self.m_mean, self.m_std = m_stats
        self.l_mean, self.l_std = l_stats

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        mel = np.load(row["mel_path"], allow_pickle=False)
        lfcc = np.load(row["lfcc_path"], allow_pickle=False)

        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)

        return (
            torch.from_numpy(mel).float(),
            torch.from_numpy(lfcc).float(),
            torch.tensor(float(row["label"]), dtype=torch.float32)
        )

# =========================================================
# 2. EER FUNCTION
# =========================================================
def compute_eer_from_scores(labels, probs):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(labels, probs)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    return fpr[idx]

# =========================================================
# 3. FINAL REPORT FUNCTION
# =========================================================
def final_forensic_report(model_path, datasets_dict, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)
    use_amp = torch.cuda.is_available()

    final_results = []

    print("\n" + "=" * 80)
    print("BSCAN FINAL FORENSIC AUDIT: PERFORMANCE SUMMARY")
    print("=" * 80)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: DataFrame is empty.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        roc_auc = roc_auc_score(all_labels, all_probs)
        preds = (all_probs > best_threshold).astype(np.int32)

        acc = accuracy_score(all_labels, preds)
        f1 = f1_score(all_labels, preds, average="macro")
        eer = compute_eer_from_scores(all_labels, all_probs)

        res = {
            "Experimental Dataset": name,
            "Samples": len(test_df),
            "Accuracy (%)": round(acc * 100, 2),
            "Macro-F1": round(f1, 4),
            "ROC-AUC": round(roc_auc, 4),
            "EER (%)": round(eer * 100, 2),
        }
        final_results.append(res)

        print(
            f"Analyzed: {name:22} | "
            f"Acc: {res['Accuracy (%)']:.2f}% | "
            f"F1: {res['Macro-F1']:.4f} | "
            f"AUC: {res['ROC-AUC']:.4f} | "
            f"EER: {res['EER (%)']:.2f}%"
        )

    report_df = pd.DataFrame(final_results)

    print("\n" + "-" * 80)
    print("TABLE III: CROSS-DATASET PERFORMANCE ANALYSIS")
    print("-" * 80)

    if len(report_df) > 0:
        display(report_df)
        report_df.to_csv("/kaggle/working/Final_Journal_Results.csv", index=False)
        print("\nSaved: /kaggle/working/Final_Journal_Results.csv")
    else:
        print("No data found to display.")

    return report_df

# =========================================================
# 4. LOAD MOZILLA TEST SET
# =========================================================
independent_test_df = pd.read_csv("/kaggle/working/metadata_dual_test_mozilla.csv").copy()

# =========================================================
# 5. FIX MENDELEY LABELS PERMANENTLY
# =========================================================
cross_test_df = pd.read_csv("/kaggle/working/metadata_dual_test_mendeley.csv").copy()
cross_test_df["label"] = 1 - cross_test_df["label"]

# Save fixed file permanently
cross_test_df.to_csv("/kaggle/working/metadata_dual_test_mendeley_fixed.csv", index=False)

print("Saved fixed Mendeley labels:")
print("/kaggle/working/metadata_dual_test_mendeley_fixed.csv")
print(cross_test_df["label"].value_counts())

# =========================================================
# 6. BUILD TEST SUITES
# =========================================================
test_suites = {
    "Mozilla Independent": independent_test_df,
    "Cross-Dataset": cross_test_df
}

# =========================================================
# 7. RUN FINAL REPORT
# =========================================================
final_report = final_forensic_report(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats
)

print("\nFinal Report:")
print(final_report)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, roc_curve, auc

# 1. Publication-quality plot setup
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 10,
})

def plot_forensic_metrics_separated(model_path, datasets_dict, m_stats, l_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    # Create 3 subplots: 1 for ROC, 2 for Confusion Matrices
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    colors = ["#1f77b4", "#ff7f0e"]

    plotted_idx = 0

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, m_stats, l_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # --- PANEL 1: ROC Curve (Shared Plot) ---
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        roc_auc = auc(fpr, tpr)
        axes[0].plot(
            fpr, tpr,
            label=f"{name}\n(AUC = {roc_auc:.4f})",
            lw=2.5,
            color=colors[min(plotted_idx, len(colors) - 1)]
        )

        # --- PANEL 2 & 3: Separate Confusion Matrices ---
        preds = (all_probs > best_threshold).astype(np.int32)
        cm = confusion_matrix(all_labels, preds, labels=[0, 1])

        row_sums = cm.sum(axis=1, keepdims=True)
        cm_perc = np.divide(
            cm * 100.0,
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0
        )

        annot_labels = np.array([
            [f"{cm[0,0]}\n({cm_perc[0,0]:.1f}%)", f"{cm[0,1]}\n({cm_perc[0,1]:.1f}%)"],
            [f"{cm[1,0]}\n({cm_perc[1,0]:.1f}%)", f"{cm[1,1]}\n({cm_perc[1,1]:.1f}%)"]
        ])

        ax_idx = plotted_idx + 1
        if ax_idx < 3:
            sns.heatmap(
                cm,
                annot=annot_labels,
                fmt="",
                cmap="Blues",
                ax=axes[ax_idx],
                xticklabels=["Real", "Fake"],
                yticklabels=["Real", "Fake"],
                annot_kws={"size": 12, "weight": "bold"},
                cbar=False
            )

            axes[ax_idx].set_title(f"Confusion Matrix:\n{name}", pad=10)
            axes[ax_idx].set_xlabel("Predicted Label", fontweight="bold")
            axes[ax_idx].set_ylabel("True Label", fontweight="bold")

        plotted_idx += 1

    # ROC final formatting
    axes[0].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
    axes[0].set_xlim([0.0, 1.0])
    axes[0].set_ylim([0.0, 1.05])
    axes[0].set_xlabel("False Positive Rate (FPR)", fontweight="bold")
    axes[0].set_ylabel("True Positive Rate (TPR)", fontweight="bold")
    axes[0].set_title("Comparative ROC Analysis", pad=15)
    axes[0].legend(loc="lower right", frameon=True)
    axes[0].grid(True, linestyle=":", alpha=0.6)

    plt.tight_layout(pad=3.0)
    plt.savefig("/kaggle/working/BSCAN_Separated_Metrics.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved: /kaggle/working/BSCAN_Separated_Metrics.png")
    print(f"Threshold used: {best_threshold:.2f}")

In [ ]:
plot_forensic_metrics_separated(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats
)

In [ ]:
import torch

# 1. PARAMETER COUNT

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable
    return total, trainable, non_trainable


# 2. LAYER COUNT

def count_layers(model):
    return sum(1 for _ in model.modules())


# 3. EXECUTION

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

total, trainable, non_trainable = count_parameters(model)
layers = count_layers(model)

# 4. MEMORY FOOTPRINT (FP32)
model_size_mb = (total * 4) / (1024**2)


# 5. PRINT REPORT

print("\n" + "="*55)
print("BSCAN MODEL COMPLEXITY AUDIT")
print("="*55)

print(f"{'Metric':30} | {'Value':>15}")
print("-" * 55)
print(f"{'Total Parameters':30} | {total:,}")
print(f"{'Trainable Parameters':30} | {trainable:,}")
print(f"{'Non-Trainable Parameters':30} | {non_trainable:,}")
print(f"{'Total Layers':30} | {layers}")
print(f"{'Memory Footprint (FP32)':30} | {model_size_mb:.2f} MB")
print("-" * 55)

# 6. EFFICIENCY CLASSIFICATION 

if total < 5_000_000:
    status = "Ultra-Lightweight (Edge/Real-Time Ready)"
elif total < 15_000_000:
    status = "Highly Efficient (Forensic Deployment Ready)"
else:
    status = "Moderate Complexity (Server-Level)"

print(f"{'Model Status':30} | {status}")
print("="*55)

data leakage check

In [ ]:
def check_data_overlap_by_source(train_csv, val_csv):
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)

    # Extract ORIGINAL FILE ID (remove _chk part)
    def get_source_id(path):
        fname = os.path.basename(path)
        return fname.split('_chk')[0]

    train_ids = train_df['mel_path'].apply(get_source_id)
    val_ids = val_df['mel_path'].apply(get_source_id)

    train_set = set(train_ids)
    val_set = set(val_ids)

    overlap = train_set.intersection(val_set)

    print("\n--- LEAKAGE CHECK (SOURCE LEVEL) ---")
    print(f"Unique sources in Train: {len(train_set)}")
    print(f"Unique sources in Val:   {len(val_set)}")
    print(f"Overlapping sources:     {len(overlap)}")

    if len(overlap) > 0:
        print(" DATA LEAKAGE DETECTED (same audio in train & val)")
        print("Sample overlaps:", list(overlap)[:10])
    else:
        print(" No leakage. Splits are clean at source level.")

# Run
check_data_overlap_by_source(
    'metadata_dual_train.csv',
    'metadata_dual_val.csv'
)

# Reliability and Statistical Significance

ROC Curve, AUC and EER

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_curve, auc

# --- 1. Plot Settings  ---
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14
})

def plot_final_roc(model_path, datasets_dict, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # 2. Model Initialization
    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    plt.figure(figsize=(9, 7))

    print("Generating Final ROC Audit...")
    print("-" * 60)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: empty dataframe.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # --- RAW ROC ONLY (NO INVERSION) ---
        fpr, tpr, _ = roc_curve(all_labels, all_probs)
        fnr = 1 - tpr
        roc_auc = auc(fpr, tpr)

        idx = np.nanargmin(np.abs(fpr - fnr))
        eer_val = (fpr[idx] + fnr[idx]) / 2.0

        plt.plot(
            fpr,
            tpr,
            lw=2.5,
            label=f"{name} (AUC: {roc_auc:.4f}, EER: {eer_val*100:.2f}%)"
        )

        print(f"{name:20} | AUC: {roc_auc:.4f} | EER: {eer_val*100:.2f}%")

    # 3. Final Plot Aesthetics
    plt.plot([0, 1], [0, 1], color="grey", lw=1, linestyle="--")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate (FPR)", fontweight="bold")
    plt.ylabel("True Positive Rate (TPR)", fontweight="bold")
    plt.title("Receiver Operating Characteristic (ROC) Forensic Audit")
    plt.legend(loc="lower right", frameon=True, shadow=True, borderpad=1)
    plt.grid(True, linestyle=":", alpha=0.7)

    # 4. Save high-resolution figure
    plt.savefig("/kaggle/working/BSCAN_Final_ROC_Journal.png", dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved: /kaggle/working/BSCAN_Final_ROC_Journal.png")

# --- 5. Execution ---
plot_final_roc(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats
)

Confidence Interval (Bootstrapping)

In [ ]:

def compute_eer_stable(labels, scores):
    """Stable EER calculation for bootstrap iterations."""
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2.0

def bootstrap_evaluation(model_path, datasets_dict, mel_stats, lfcc_stats, n_bootstraps=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # 1. Initialize model
    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Starting Statistical Bootstrapping ({n_bootstraps} iterations)...")
    print("-" * 75)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: empty dataframe.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        # 2. Inference pass
        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # Raw point estimates
        point_auc = roc_auc_score(all_labels, all_probs)
        point_eer = compute_eer_stable(all_labels, all_probs)

        bootstrapped_auc = []
        bootstrapped_eer = []

        # 3. Bootstrapping
        rng = np.random.RandomState(42)
        indices = rng.randint(0, len(all_probs), size=(n_bootstraps, len(all_probs)))

        for i in tqdm(range(n_bootstraps), desc=f"Analyzing {name}"):
            resample_idx = indices[i]

            if len(np.unique(all_labels[resample_idx])) < 2:
                continue

            auc_boot = roc_auc_score(all_labels[resample_idx], all_probs[resample_idx])
            eer_boot = compute_eer_stable(all_labels[resample_idx], all_probs[resample_idx])

            bootstrapped_auc.append(auc_boot)
            bootstrapped_eer.append(eer_boot)

        bootstrapped_auc = np.array(bootstrapped_auc, dtype=np.float32)
        bootstrapped_eer = np.array(bootstrapped_eer, dtype=np.float32)

        if len(bootstrapped_auc) == 0 or len(bootstrapped_eer) == 0:
            print(f"Skipping CI report for {name}: no valid bootstrap samples.")
            continue

        auc_ci_low, auc_ci_high = np.percentile(bootstrapped_auc, [2.5, 97.5])
        eer_ci_low, eer_ci_high = np.percentile(bootstrapped_eer, [2.5, 97.5])

        print(f"\nStatistical Analysis Report for {name}:")
        print(f"{'Metric':12} | {'Point Estimate':14} | {'Bootstrap Mean':14} | {'95% Confidence Interval'}")
        print("-" * 90)
        print(f"{'AUC':12} | {point_auc:.4f}         | {bootstrapped_auc.mean():.4f}         | ({auc_ci_low:.4f} - {auc_ci_high:.4f})")
        print(f"{'EER (%)':12} | {point_eer*100:.2f}%       | {bootstrapped_eer.mean()*100:.2f}%       | ({eer_ci_low*100:.2f}% - {eer_ci_high*100:.2f}%)")
        print("=" * 90)

# --- EXECUTION ---
bootstrap_evaluation(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats,
    n_bootstraps=1000
)

BSCAN (No-Attention) Code

In [ ]:
def bootstrap_no_attention_report(model_path, datasets_dict, mel_stats, lfcc_stats, n_bootstraps=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # Initialize with attention DISABLED
    model = BSCAN(use_attention=False, mode="dual", hidden_dim=64).to(device)

    # Load weights (strict=False because attention layers are skipped)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"], strict=False)
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    print("ABLATION STUDY: BSCAN (No-Attention) STATISTICAL AUDIT")
    print("=" * 75)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: empty dataframe.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        metrics = {"Accuracy": [], "F1-Score": [], "AUC": [], "EER": []}

        rng = np.random.RandomState(42)
        indices = rng.randint(0, len(all_probs), size=(n_bootstraps, len(all_probs)))

        for i in tqdm(range(n_bootstraps), desc=f"Ablation Analysis: {name}"):
            idx = indices[i]

            if len(np.unique(all_labels[idx])) < 2:
                continue

            probs_boot = all_probs[idx]
            labels_boot = all_labels[idx]
            preds_boot = (probs_boot > best_threshold).astype(np.int32)

            metrics["Accuracy"].append(accuracy_score(labels_boot, preds_boot))
            metrics["F1-Score"].append(f1_score(labels_boot, preds_boot, average="macro"))
            metrics["AUC"].append(roc_auc_score(labels_boot, probs_boot))
            metrics["EER"].append(compute_eer_stable(labels_boot, probs_boot))

        print(f"\nNo-Attention Results for {name}:")
        print(f"{'Metric':12} | {'Mean Value':14} | {'95% Confidence Interval'}")
        print("-" * 75)

        for metric_name, values in metrics.items():
            values = np.array(values, dtype=np.float32)

            if len(values) == 0:
                print(f"{metric_name:12} | No valid bootstrap samples")
                continue

            low, high = np.percentile(values, [2.5, 97.5])
            mean_v = values.mean()

            if metric_name in ["Accuracy", "EER"]:
                print(f"{metric_name:12} | {mean_v*100:10.2f}% | ({low*100:6.2f}% - {high*100:6.2f}%)")
            else:
                print(f"{metric_name:12} | {mean_v:14.4f} | ({low:6.4f} - {high:6.4f})")

        print("=" * 75)

# --- RUN NO-ATTENTION ANALYSIS ---
bootstrap_no_attention_report(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats,
    n_bootstraps=1000
)

Single vs Dual Branch

Single Branch (Mel-only)

In [ ]:
import torch
import numpy as np
import copy
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def evaluate_single_branch_mel(model_path, datasets_dict, mel_stats, lfcc_stats, n_bootstraps=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # 1. Initialize in Mel-only mode
    model = BSCAN(use_attention=True, mode="mel_only", hidden_dim=64).to(device)

    # 2. Load and prepare checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = copy.deepcopy(checkpoint["model_state_dict"])

    # Slice classifier weights to isolate Mel branch
    if "classifier.0.weight" in state_dict:
        original_weight = state_dict["classifier.0.weight"]
        if original_weight.shape[1] == 128:
            print("Adjusting classifier weights to isolate Mel branch...")
            state_dict["classifier.0.weight"] = original_weight[:, :64]

    model.load_state_dict(state_dict, strict=False)
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    print("ABLATION AUDIT: SINGLE-BRANCH (MEL-ONLY)")
    print("=" * 75)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: empty dataframe.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, _, labels in loader:
                mels = mels.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, None).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        metrics = {"Accuracy": [], "F1-Score": [], "AUC": [], "EER": []}

        rng = np.random.RandomState(42)
        indices = rng.randint(0, len(all_probs), size=(n_bootstraps, len(all_probs)))

        for i in tqdm(range(n_bootstraps), desc=f"Mel-only: {name}"):
            idx = indices[i]

            if len(np.unique(all_labels[idx])) < 2:
                continue

            probs_boot = all_probs[idx]
            labels_boot = all_labels[idx]
            preds_boot = (probs_boot > best_threshold).astype(np.int32)

            metrics["Accuracy"].append(accuracy_score(labels_boot, preds_boot))
            metrics["F1-Score"].append(f1_score(labels_boot, preds_boot, average="macro"))
            metrics["AUC"].append(roc_auc_score(labels_boot, probs_boot))
            metrics["EER"].append(compute_eer_stable(labels_boot, probs_boot))

        print(f"\nMel-only Results for {name}:")
        print(f"{'Metric':12} | {'Mean Value':14} | {'95% Confidence Interval'}")
        print("-" * 75)

        for metric_name, values in metrics.items():
            values = np.array(values, dtype=np.float32)

            if len(values) == 0:
                print(f"{metric_name:12} | No valid bootstrap samples")
                continue

            low, high = np.percentile(values, [2.5, 97.5])
            mean_v = values.mean()

            if metric_name in ["Accuracy", "EER"]:
                print(f"{metric_name:12} | {mean_v*100:10.2f}% | ({low*100:6.2f}% - {high*100:6.2f}%)")
            else:
                print(f"{metric_name:12} | {mean_v:14.4f} | ({low:6.4f} - {high:6.4f})")

        print("=" * 75)

# --- EXECUTION ---
evaluate_single_branch_mel(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats,
    n_bootstraps=1000
)

Single Branch (LFCC only)

In [ ]:
# Required helper
def compute_eer_stable(labels, scores):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(labels, scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fpr - fnr))
    return (fpr[idx] + fnr[idx]) / 2.0

def evaluate_single_branch_mel(model_path, datasets_dict, mel_stats, lfcc_stats, n_bootstraps=1000):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # 1. Initialize in Mel-only mode
    model = BSCAN(use_attention=True, mode="mel_only", hidden_dim=64).to(device)

    # 2. Load and prepare checkpoint
    checkpoint = torch.load(model_path, map_location=device)
    state_dict = copy.deepcopy(checkpoint["model_state_dict"])

    # Slice classifier weights to isolate Mel branch
    if "classifier.0.weight" in state_dict:
        original_weight = state_dict["classifier.0.weight"]
        if original_weight.shape[1] == 128:
            print("Adjusting classifier weights to isolate Mel branch...")
            state_dict["classifier.0.weight"] = original_weight[:, :64]

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    model.eval()

    if missing:
        print("Missing keys (expected for ablation):", len(missing))
    if unexpected:
        print("Unexpected keys:", len(unexpected))

    best_threshold = checkpoint.get("best_threshold", 0.5)

    print("ABLATION AUDIT: SINGLE-BRANCH (MEL-ONLY)")
    print("=" * 90)

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            print(f"Skipping {name}: empty dataframe.")
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, _, labels in loader:
                mels = mels.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, None).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        all_probs = np.array(all_probs, dtype=np.float32)
        all_labels = np.array(all_labels, dtype=np.int32)

        if len(np.unique(all_labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # Point estimates
        point_preds = (all_probs > best_threshold).astype(np.int32)
        point_acc = accuracy_score(all_labels, point_preds)
        point_f1 = f1_score(all_labels, point_preds, average="macro")
        point_auc = roc_auc_score(all_labels, all_probs)
        point_eer = compute_eer_stable(all_labels, all_probs)

        metrics = {"Accuracy": [], "F1-Score": [], "AUC": [], "EER": []}

        rng = np.random.RandomState(42)
        indices = rng.randint(0, len(all_probs), size=(n_bootstraps, len(all_probs)))

        for i in tqdm(range(n_bootstraps), desc=f"Mel-only: {name}"):
            idx = indices[i]

            if len(np.unique(all_labels[idx])) < 2:
                continue

            probs_boot = all_probs[idx]
            labels_boot = all_labels[idx]
            preds_boot = (probs_boot > best_threshold).astype(np.int32)

            metrics["Accuracy"].append(accuracy_score(labels_boot, preds_boot))
            metrics["F1-Score"].append(f1_score(labels_boot, preds_boot, average="macro"))
            metrics["AUC"].append(roc_auc_score(labels_boot, probs_boot))
            metrics["EER"].append(compute_eer_stable(labels_boot, probs_boot))

        print(f"\nMel-only Results for {name}:")
        print(f"{'Metric':12} | {'Point Est.':12} | {'Bootstrap Mean':14} | {'95% Confidence Interval'}")
        print("-" * 90)

        point_lookup = {
            "Accuracy": point_acc,
            "F1-Score": point_f1,
            "AUC": point_auc,
            "EER": point_eer
        }

        for metric_name, values in metrics.items():
            values = np.array(values, dtype=np.float32)

            if len(values) == 0:
                print(f"{metric_name:12} | No valid bootstrap samples")
                continue

            low, high = np.percentile(values, [2.5, 97.5])
            mean_v = values.mean()
            point_v = point_lookup[metric_name]

            if metric_name in ["Accuracy", "EER"]:
                print(
                    f"{metric_name:12} | {point_v*100:8.2f}%   | {mean_v*100:10.2f}% | "
                    f"({low*100:6.2f}% - {high*100:6.2f}%)"
                )
            else:
                print(
                    f"{metric_name:12} | {point_v:10.4f} | {mean_v:14.4f} | "
                    f"({low:6.4f} - {high:6.4f})"
                )

        print("=" * 90)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# --- EXECUTION ---
evaluate_single_branch_mel(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats,
    n_bootstraps=1000
)

Mann-Whitney U-test,Z-Test Result, Tandem Decision Cost Function (t-DCF)

In [ ]:
import torch
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.proportion import proportions_ztest
from torch.utils.data import DataLoader

def perform_final_statistical_audit(model_path, datasets_dict, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)

    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    audit_data = {}

    print("Gathering Inference Data for Advanced Stats...")

    for name, test_df in datasets_dict.items():
        if test_df is None or test_df.empty:
            continue

        loader = DataLoader(
            BengaliDeepfakeDataset_Direct(test_df, mel_stats, lfcc_stats),
            batch_size=32,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        all_probs, all_labels = [], []

        with torch.no_grad():
            for mels, lfccs, labels in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    probs = torch.sigmoid(outputs)

                all_probs.extend(probs.cpu().numpy().tolist())
                all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

        probs_np = np.array(all_probs, dtype=np.float32)
        labels_np = np.array(all_labels, dtype=np.int32)

        if len(np.unique(labels_np)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        audit_data[name] = {"probs": probs_np, "labels": labels_np}

    print("\n" + "=" * 60)
    print("ADVANCED FORENSIC STATISTICAL REPORT")
    print("=" * 60)

    # 1. Mann-Whitney U-Test (Class Separation)
    for name, data in audit_data.items():
        real_scores = data["probs"][data["labels"] == 0]
        fake_scores = data["probs"][data["labels"] == 1]

        u_stat, p_val = mannwhitneyu(real_scores, fake_scores, alternative="two-sided")

        print(f"Dataset: {name}")
        print(f"  Separation Significance (p-value): {p_val:.4e}")
        if p_val < 0.001:
            print("  Result: Highly significant class separation.")
        elif p_val < 0.05:
            print("  Result: Significant class separation.")
        else:
            print("  Result: No statistically significant separation.")
        print("-" * 50)

    # 2. Two-Proportion Z-Test (Generalization Gap)
    if "Mozilla Independent" in audit_data and "Cross-Dataset" in audit_data:
        m = audit_data["Mozilla Independent"]
        c = audit_data["Cross-Dataset"]

        # Use the checkpoint threshold if available
        threshold = checkpoint.get("best_threshold", 0.5)

        succ_m = np.sum((m["probs"] > threshold).astype(np.int32) == m["labels"])
        succ_c = np.sum((c["probs"] > threshold).astype(np.int32) == c["labels"])

        z_stat, z_pval = proportions_ztest(
            [succ_m, succ_c],
            [len(m["labels"]), len(c["labels"])]
        )

        print("Generalization Gap Audit (Z-Test)")
        print(f"  Internal Accuracy: {succ_m / len(m['labels']) * 100:.2f}%")
        print(f"  External Accuracy: {succ_c / len(c['labels']) * 100:.2f}%")
        print(f"  Z-Statistic: {z_stat:.4f}")
        print(f"  P-Value:     {z_pval:.4e}")
        print("=" * 60)

# --- EXECUTION ---
perform_final_statistical_audit(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats
)

Noise Robustness Stress Test (SNR Audit)

In [ ]:

class SpectralMaskDataset(Dataset):
    def __init__(self, df, mel_stats, lfcc_stats, mask_prob=0.0):
        self.df = df.reset_index(drop=True)
        self.m_mean, self.m_std = mel_stats
        self.l_mean, self.l_std = lfcc_stats
        self.mask_prob = mask_prob  # probability of masking a frequency bin

        cols = self.df.columns.tolist()
        self.mel_col = next((c for c in cols if "mel" in c.lower() and "path" in c.lower()), "mel_path")
        self.lfcc_col = next((c for c in cols if "lfcc" in c.lower() and "path" in c.lower()), "lfcc_path")
        self.label_col = next((c for c in cols if "label" in c.lower() or "class" in c.lower()), "label")

    def __len__(self):
        return len(self.df)

    def apply_spectral_mask(self, feat):
        """
        feat shape: [C, F, T]
        Masks frequency bins consistently across time and channels.
        """
        if self.mask_prob <= 0:
            return feat

        keep_mask = (torch.rand(feat.shape[1]) > self.mask_prob).float()   # [F]
        keep_mask = keep_mask.unsqueeze(0).unsqueeze(-1)                   # [1, F, 1]
        return feat * keep_mask

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        mel = torch.from_numpy(np.load(row[self.mel_col], allow_pickle=False)).float()
        lfcc = torch.from_numpy(np.load(row[self.lfcc_col], allow_pickle=False)).float()

        # Mask first, then normalize
        mel = self.apply_spectral_mask(mel)
        lfcc = self.apply_spectral_mask(lfcc)

        mel = (mel - self.m_mean) / (self.m_std + 1e-8)
        lfcc = (lfcc - self.l_mean) / (self.l_std + 1e-8)

        label = torch.tensor(float(row[self.label_col]), dtype=torch.float32)
        return mel, lfcc, label


def run_masking_audit(model_path, train_df, test_suites, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    # 0% (clean), 10%, 30%, 50%
    mask_levels = [0.0, 0.1, 0.3, 0.5]

    scenarios = {
        "1. Training (Sample)": train_df.sample(n=min(1000, len(train_df)), random_state=42).copy(),
        "2. Internal (Mozilla)": test_suites.get("Mozilla Independent"),
        "3. External (Cross)": test_suites.get("Cross-Dataset"),
    }

    print("BSCAN ROBUSTNESS: SPECTRAL MASKING AUDIT")
    print("=" * 95)

    for name, df in scenarios.items():
        if df is None or df.empty:
            continue

        print(f"\nSCENARIO: {name}")
        print(f"{'Data Loss %':12} | {'Accuracy':10} | {'AUC-ROC':10} | {'Status'}")
        print("-" * 80)

        for m_lvl in mask_levels:
            dataset = SpectralMaskDataset(df, mel_stats, lfcc_stats, mask_prob=m_lvl)
            loader = DataLoader(
                dataset,
                batch_size=64,
                shuffle=False,
                num_workers=0,
                pin_memory=torch.cuda.is_available()
            )

            all_probs, all_labels = [], []

            with torch.no_grad():
                for mels, lfccs, labels in loader:
                    mels = mels.to(device, non_blocking=True)
                    lfccs = lfccs.to(device, non_blocking=True)

                    with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                        outputs = model(mels, lfccs).view(-1)
                        probs = torch.sigmoid(outputs)

                    all_probs.extend(probs.cpu().numpy().tolist())
                    all_labels.extend(labels.cpu().numpy().astype(np.int32).tolist())

            all_probs = np.array(all_probs, dtype=np.float32)
            all_labels = np.array(all_labels, dtype=np.int32)

            if len(np.unique(all_labels)) < 2:
                print(f"{int(m_lvl*100):>3}%         | Skipped (single class)")
                continue

            preds = (all_probs > best_threshold).astype(np.int32)
            acc = accuracy_score(all_labels, preds)
            auc_val = roc_auc_score(all_labels, all_probs)

            status = "Robust" if auc_val > 0.80 else "Decaying" if auc_val > 0.70 else "Sensitive"
            lvl_name = f"{int(m_lvl * 100)}%"

            print(f"{lvl_name:12} | {acc*100:9.2f}% | {auc_val:10.4f} | {status}")

        print("=" * 95)

# --- EXECUTION ---
run_masking_audit(
    "/kaggle/working/BSCAN_Best_Model.pth",
    train_df,
    test_suites,   # make sure Cross-Dataset here is your fixed Mendeley labels
    mel_stats,
    lfcc_stats
)

Short-Duration Analysis

In [ ]:


def run_global_duration_audit(model_path, train_df, test_suites, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    # Bins for forensic analysis
    bins = [0, 2, 4, 7, np.inf]
    labels = ["Short (<2s)", "Medium (2-4s)", "Standard (4-7s)", "Long (>7s)"]

    # Define the 3 master sets
    all_sets = {
        "1. Training Set (Mozilla)": train_df.copy(),
        "2. Internal Validation (Mozilla)": test_suites.get("Mozilla Independent"),
        "3. External Cross-Dataset (Unseen)": test_suites.get("Cross-Dataset"),
    }

    print("BSCAN GLOBAL TEMPORAL SENSITIVITY REPORT")
    print("=" * 100)

    for ds_name, df in all_sets.items():
        if df is None or df.empty:
            continue

        df = df.copy()

        # Calculate duration if not present
        if "duration" not in df.columns:
            # Mel shape is [C, F, T], so time frames are shape[2]
            # duration_sec ~= (T * hop_length) / sample_rate
            sample_mel = np.load(df["mel_path"].iloc[0], allow_pickle=False)
            if sample_mel.ndim != 3:
                raise ValueError(f"Unexpected mel shape: {sample_mel.shape}")

            time_frames = sample_mel.shape[2]
            hop_length = 512
            sample_rate = 16000
            est_duration = (time_frames * hop_length) / sample_rate

            # In your pipeline, most saved features are fixed-length chunks,
            # so this duration will usually be identical across rows.
            df["duration"] = est_duration

        df["duration_cat"] = pd.cut(df["duration"], bins=bins, labels=labels)

        print(f"\nSCENARIO: {ds_name}")
        print(f"{'Category':18} | {'N':6} | {'Avg Dur':8} | {'Accuracy':10} | {'AUC-ROC'}")
        print("-" * 80)

        for cat in labels:
            sub_df = df[df["duration_cat"] == cat].copy()
            if len(sub_df) < 2:
                continue

            dataset = BengaliDeepfakeDataset_Direct(sub_df, mel_stats, lfcc_stats)
            loader = DataLoader(
                dataset,
                batch_size=64,
                shuffle=False,
                num_workers=0,
                pin_memory=torch.cuda.is_available()
            )

            probs, trues = [], []

            with torch.no_grad():
                for mels, lfccs, labs in loader:
                    mels = mels.to(device, non_blocking=True)
                    lfccs = lfccs.to(device, non_blocking=True)

                    with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                        outputs = model(mels, lfccs).view(-1)
                        batch_probs = torch.sigmoid(outputs)

                    probs.extend(batch_probs.cpu().numpy().tolist())
                    trues.extend(labs.cpu().numpy().astype(np.int32).tolist())

            probs = np.array(probs, dtype=np.float32)
            trues = np.array(trues, dtype=np.int32)

            if len(np.unique(trues)) < 2:
                continue

            acc = accuracy_score(trues, (probs > best_threshold).astype(np.int32))
            auc_val = roc_auc_score(trues, probs)
            avg_d = sub_df["duration"].mean()

            print(f"{cat:18} | {len(sub_df):6} | {avg_d:6.2f}s | {acc*100:9.2f}% | {auc_val:10.4f}")

        print("=" * 100)

# --- EXECUTION ---
run_global_duration_audit(
    "/kaggle/working/BSCAN_Best_Model.pth",
    train_df,
    test_suites,
    mel_stats,
    lfcc_stats
)

Optimal Threshold Calibration

In [ ]:
import numpy as np
import torch
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from torch.utils.data import DataLoader

def run_corrected_audit(model_path, test_suites, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Focus on Cross-Dataset
    df = test_suites.get("Cross-Dataset")
    if df is None or df.empty:
        print("Cross-Dataset not found in test_suites.")
        return

    dataset = BengaliDeepfakeDataset_Direct(df, mel_stats, lfcc_stats)
    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    probs, labels = [], []

    with torch.no_grad():
        for mels, lfccs, labs in loader:
            mels = mels.to(device, non_blocking=True)
            lfccs = lfccs.to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                outputs = model(mels, lfccs).view(-1)
                batch_probs = torch.sigmoid(outputs)

            probs.extend(batch_probs.cpu().numpy().tolist())
            labels.extend(labs.cpu().numpy().astype(np.int32).tolist())

    probs = np.array(probs, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)

    if len(np.unique(labels)) < 2:
        print("Cross-Dataset has only one class present.")
        return

    # Raw AUC
    current_auc = roc_auc_score(labels, probs)

    # EER
    fpr, tpr, thresholds = roc_curve(labels, probs)
    eer = brentq(lambda x: 1.0 - x - interp1d(fpr, tpr)(x), 0.0, 1.0)
    opt_threshold = float(interp1d(fpr, thresholds)(eer))

    print("\nFINAL CALIBRATED RESULTS (CROSS-DATASET)")
    print("=" * 60)
    print(f"Raw AUC-ROC:           {current_auc:.4f}")
    print(f"Raw EER:               {eer*100:.2f}%")
    print(f"Optimal Threshold:     {opt_threshold:.4f}")
    print("=" * 60)

    if current_auc > 0.85:
        print("Strong cross-dataset robustness.")
    elif current_auc > 0.70:
        print("Moderate cross-dataset robustness.")
    else:
        print("Cross-dataset performance is weak. Consider retraining or domain adaptation.")

# --- EXECUTION ---
run_corrected_audit(
    "/kaggle/working/BSCAN_Best_Model.pth",
    test_suites,
    mel_stats,
    lfcc_stats
)

Visual Proof(Loss and AUC Curves)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
import torch
import numpy as np
from torch.utils.data import DataLoader

# Set journal-level plotting style
plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "figure.dpi": 300,
    "font.family": "serif"
})

def generate_visual_proof_tiered(model_path, val_df, test_suites, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # Load model
    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    best_threshold = checkpoint.get("best_threshold", 0.5)

    # Define the 3 tiers of unseen data
    datasets = {
        "Tier 1: Internal Validation": val_df,
        "Tier 2: Independent Mozilla": test_suites.get("Mozilla Independent"),
        "Tier 3: External Cross-Dataset": test_suites.get("Cross-Dataset")
    }

    # Setup figures
    fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
    fig_cm, axes_cm = plt.subplots(1, 3, figsize=(18, 5))

    colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

    print("Generating Tiered Visual Proofs (Validation Focus)...")
    print("-" * 60)

    for i, (name, df) in enumerate(datasets.items()):
        if df is None or df.empty:
            print(f"Skipping {name}: DataFrame not found or empty.")
            continue

        # Inference dataset
        dataset = BengaliDeepfakeDataset_Direct(df, mel_stats, lfcc_stats)
        loader = DataLoader(
            dataset,
            batch_size=64,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        probs, labels = [], []

        with torch.no_grad():
            for mels, lfccs, labs in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    batch_probs = torch.sigmoid(outputs)

                probs.extend(batch_probs.cpu().numpy().tolist())
                labels.extend(labs.cpu().numpy().astype(np.int32).tolist())

        probs = np.array(probs, dtype=np.float32)
        labels = np.array(labels, dtype=np.int32)

        if len(np.unique(labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # ROC
        fpr, tpr, _ = roc_curve(labels, probs)
        roc_auc = auc(fpr, tpr)

        ax_roc.plot(
            fpr, tpr,
            color=colors[i],
            lw=2.5,
            label=f"{name} (AUC = {roc_auc:.4f})"
        )

        # Confusion Matrix using saved threshold
        preds = (probs > best_threshold).astype(np.int32)
        cm = confusion_matrix(labels, preds, labels=[0, 1])

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=["Real", "Fake"]
        )
        disp.plot(ax=axes_cm[i], cmap="Blues", colorbar=False)
        axes_cm[i].set_title(name, fontweight="bold")

    # Finalize ROC plot
    ax_roc.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--")
    ax_roc.set_xlim([0.0, 1.0])
    ax_roc.set_ylim([0.0, 1.05])
    ax_roc.set_xlabel("False Positive Rate (FPR)")
    ax_roc.set_ylabel("True Positive Rate (TPR)")
    ax_roc.set_title("Figure 4: Tiered ROC Comparison (Generalization Capacity)")
    ax_roc.legend(loc="lower right", frameon=True)
    ax_roc.grid(alpha=0.2)
    fig_roc.savefig("/kaggle/working/BSCAN_Tiered_ROC.png", bbox_inches="tight")

    # Finalize confusion matrices
    fig_cm.suptitle(
        "Figure 5: Confusion Matrices Across Generalization Tiers",
        fontsize=16,
        fontweight="bold"
    )
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig_cm.savefig("/kaggle/working/BSCAN_Tiered_CM.png", bbox_inches="tight")

    plt.show()
    print("-" * 60)
    print("Saved:")
    print("/kaggle/working/BSCAN_Tiered_ROC.png")
    print("/kaggle/working/BSCAN_Tiered_CM.png")

# --- EXECUTION ---
generate_visual_proof_tiered(
    "/kaggle/working/BSCAN_Best_Model.pth",
    val_df,
    test_suites,
    mel_stats,
    lfcc_stats
)

In [ ]:
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import roc_curve, auc
from torch.utils.data import DataLoader

def generate_tiered_text_report(model_path, val_df, test_suites, mel_stats, lfcc_stats):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # Load model
    model = BSCAN(use_attention=True, mode="dual", hidden_dim=64).to(device)
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # Use saved threshold if available
    best_threshold = checkpoint.get("best_threshold", 0.5)

    # Define tiers
    datasets = {
        "Tier 1: Internal Validation": val_df,
        "Tier 2: Independent Mozilla": test_suites.get("Mozilla Independent"),
        "Tier 3: External Cross-Dataset": test_suites.get("Cross-Dataset")
    }

    report_rows = []

    print("GENERATING FORMAL PERFORMANCE REPORT...")
    print("=" * 90)

    for name, df in datasets.items():
        if df is None or df.empty:
            continue

        # Inference
        dataset = BengaliDeepfakeDataset_Direct(df, mel_stats, lfcc_stats)
        loader = DataLoader(
            dataset,
            batch_size=64,
            shuffle=False,
            num_workers=0,
            pin_memory=torch.cuda.is_available()
        )

        probs, labels = [], []

        with torch.no_grad():
            for mels, lfccs, labs in loader:
                mels = mels.to(device, non_blocking=True)
                lfccs = lfccs.to(device, non_blocking=True)

                with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                    outputs = model(mels, lfccs).view(-1)
                    batch_probs = torch.sigmoid(outputs)

                probs.extend(batch_probs.cpu().numpy().tolist())
                labels.extend(labs.cpu().numpy().astype(np.int32).tolist())

        probs = np.array(probs, dtype=np.float32)
        labels = np.array(labels, dtype=np.int32)

        if len(np.unique(labels)) < 2:
            print(f"Skipping {name}: only one class present.")
            continue

        # Metrics
        fpr, tpr, thresholds = roc_curve(labels, probs)
        roc_auc = auc(fpr, tpr)

        fnr = 1 - tpr
        eer_idx = np.nanargmin(np.abs(fnr - fpr))
        eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0

        preds = (probs >= best_threshold).astype(np.int32)
        accuracy = (preds == labels).mean()

        report_rows.append({
            "Evaluation Tier": name,
            "AUC-ROC": round(float(roc_auc), 4),
            "EER (%)": round(float(eer * 100), 2),
            "Accuracy (%)": round(float(accuracy * 100), 2),
            "Threshold": round(float(best_threshold), 4),
            "Sample Count": int(len(df))
        })

    report_df = pd.DataFrame(report_rows)
    print(report_df.to_string(index=False))
    print("=" * 90)

    # Manuscript text block
    print("\nMANUSCRIPT TEXT (RESULTS SECTION):")
    print("-" * 40)

    if len(report_rows) >= 3:
        t1_auc = report_rows[0]["AUC-ROC"]
        t2_auc = report_rows[1]["AUC-ROC"]
        t3_auc = report_rows[2]["AUC-ROC"]

        print("The proposed BSCAN architecture demonstrated strong generalization across progressively more challenging evaluation tiers.")
        print(f"In Tier 1 (Internal Validation), the model achieved an AUC-ROC of {t1_auc:.4f}.")
        print(f"In Tier 2 (Independent Mozilla), the performance remained high with an AUC-ROC of {t2_auc:.4f}.")
        print(f"In Tier 3 (External Cross-Dataset), the model achieved an AUC-ROC of {t3_auc:.4f}, indicating robust cross-dataset discrimination under unseen recording conditions.")
    else:
        print("Insufficient completed tiers were available to generate the summary paragraph.")

    print("-" * 40)

    return report_df

# --- EXECUTION ---
final_report = generate_tiered_text_report(
    "/kaggle/working/BSCAN_Best_Model.pth",
    val_df,
    test_suites,
    mel_stats,
    lfcc_stats
)